# **MODULE 01 — Individual activity profiles**

This module returns synthetic individual activity profiles, starting from
the aggregated Italian (ISTAT) Time-Use-Survey inputs. It calculates
transition matrices and initial probabilities (following I-order Markov Chain method),
creates diagnostic plots, generates annual synthetic activity profiles, 
and extracts selected weeks.

The geographical aggregation used for matrix estimation is explicit and hybrid:
each individual class is assigned either region-specific matrices or one of the
defined pooled-area policies. The same mapping is used when annual profiles load
their matrices.

**Recommended steps** First input reconstruction --> Matrices --> Annual profiles --> Weekly profiles.
Running these steps together might be too demanding for allocated memory in the kernel, so it's suggested
to compute them in different runs.

**Downstream contract:** annual and weekly file names, worksheet names, activity
codes, and output folders remain compatible with the subsequent HAGRID modules.

## 1.1 Import libraries

In [ ]:
import datetime
import gc
import gzip
import pickle
from functools import lru_cache
from itertools import product
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm



# Get the start time
start_time = datetime.datetime.now()
print(f"Start time : {start_time}")


## 1.2 Selectable sections

Select **one stage group at a time**. `RUN_MATRICES` and `RUN_PLOTS` are intentionally treated as one group because they use the same filtered source datasets. If flags from two different groups are enabled, execution stops before any large input is loaded.


In [ ]:
# -----------------------------------------------------------------------------
#   Sec 1.2 -- 1)  MAIN INDEPENDENT EXECUTION STAGES
# -----------------------------------------------------------------------------
# Keep only one stage group active at a time. RUN_MATRICES and RUN_PLOTS belong
# to the same stage and may therefore both be True.

# (Optional) Construction of aggregated datasets
RUN_AGGR_DATASETS       = False         # Sec 1.8 (optional): whether you want to aggregate original TUS data in "aggregated" classes of activities
RUN_DEBUG_INPUTS        = False         # Sec 1.9 (optional): whether you want to compare aggregated dataframes with "original" one generated by the authors

# 0)
RUN_MATRICES            = False         # Sec 1.10: creation of transition matrices for several individual classes
RUN_PLOTS               = False          # Sec 1.10: plot activity profiles and consumption

# 1)
RUN_YEARLY_PROFILES     = True         # Sec 1.11: full-year activity and place profiles for individual classes
RUN_WEEKLY_PROFILES     = True         # Sec 1.12: activity profiles for individual classes for selected weeks of the year





# -----------------------------------------------------------------------------
#   Sec 1.2 -- 2) SECONDARY - DETAILED SELECTABLE SECTIONS
# -----------------------------------------------------------------------------

# Detailed choices within the matrix/plot stage
BUILD_ACTIVITY_MATRICES        = True              # Sec 1.10: calculate transition matrices for activity profiles
BUILD_PLACE_MATRICES           = True              # Sec 1.10: calculate transition matrices for occupied-place profiles
PLOT_ACTIVITIES                = True              # Sec 1.10: generate plots of activity distributions
PLOT_PLACES                    = False             # Sec 1.10: generate plots of occupied-place distributions
PLOT_LAYOUTS                   = ("two_column",)   # Sec 1.10: select one or more plot layouts: "single_column", "two_column"
INCLUDE_AREA_IN_PLOT_FILENAME  = True              # Sec 1.10: include the geographical area in plot filenames to avoid overwriting

# Optional input validation
RUN_STRICT_ROWSET_DEBUG        = False             # Sec 1.9: compare rebuilt and legacy inputs also when row order differs
AUTO_REBUILD_MISSING_INPUTS    = False             # Sec 1.10: automatically rebuild missing input files instead of raising an error

# Matrix-policy controls
MATRIX_POLICY_OVERRIDE         = None              # Sec 1.10/11: force one geographical policy for all classes; "None" uses the hybrid configuration
MIN_SOURCE_RESPONDENTS_WARNING = 5                 # Sec 1.10: warn when a filtered case contains fewer respondents than this threshold
SAVE_CASE_SUMMARY              = True              # Sec 1.10: save a CSV summary of sample sizes and geographical policies

# Annual simulation controls
RANDOM_SEED                    = None              # Sec 1.11: set an integer for reproducible stochastic profiles; "None" gives new realizations
CONTINUE_STATE_ACROSS_DAYS     = True              # Sec 1.11: use the final state of each day as the initial state of the following day





# -----------------------------------------------------------------------------
#   Sec 1.2 -- 3) OPTIONAL FILTERS FOR ACTIVITY PROFILE PLOT EXAMPLE
# -----------------------------------------------------------------------------
# (if RUN_PLOTS = True) Optional filters for plot generation; None means all available cases
PLOT_SELECTED_CONFIGURATIONS = ((0, 0),)          # individual classes to plot; e.g. ((0, 0), (1, 1)) (check description of individual classes in Section 1.3); use "None" to select all classes
PLOT_SELECTED_AREAS          = ("Cen",)            # matrix areas to plot; e.g. ("Cen",) or ("N-O_N-E_Cen",); use "None" to select all areas
PLOT_SELECTED_QUARTERS       = ("Ago-Set-Ott",)    # quarters to plot: Nov-Dic-Gen, Feb-Mar-Apr, Mag-Giu-Lug, Ago-Set-Ott
PLOT_SELECTED_DAY_TYPES      = ("domenica",)       # day types to plot: feriale, sabato, domenica
PLOT_SAVE_PNG                = True                # save plots as PNG
PLOT_SAVE_PDF                = False               # save plots as PDF


## 1.3 Dictionaries and source-data labels

### 1.3.1 Main input data

In [ ]:
# -------------------------------------------------------------------------
#   Sec 1.3.1 -- 1) DICTIONARIES FOR INPUT DATA
# -------------------------------------------------------------------------

"""
    ---    DESCRIPTION  ---

    Below are the values for household type and individual position within the household used in Time-Use-Survey data

    TIPONUCLEO_LIST:

    tipnu2	household type      class

    0	    0                   Single members
    1	    1                   Couple with children
    2	    2                   Couple without children
    3	    3                   Male single parent
    4	    4                   Female single parent


    POSNUCLEO_LIST:

    rpnuc2	position within the household

    0	    Single member
    1	    Household head
    2	    Spouse
    3	    Child

    Example:

    household_type_list = [1] --> Couple with children
    household_position_list = [3] --> Child

    WARNING: since the original used dataset is for the Italian case, in the following activities and location are referred to Italian classes. In case of EUROSTAT data, you need to align the categories. In # you can find the corresponding english name for activities and locations.

"""


 # Definition of the stages inside module 01
STAGE_LABELS = {
    "optional_rebuild": "1.8 OPTIONAL - rebuild aggregated inputs",
    "optional_debug": "1.9 OPTIONAL - debug aggregated inputs",
    "matrices_plots": "1.10 matrices and/or plots",
    "profile_pipeline": "1.11 annual profiles and/or 1.12 weekly extraction",
}


# These values are intentionally kept in Italian because they are ISTAT source-data labels.
aggregated_activities = [
    'Dormire/Riposo', 'Igiene e vestirsi', 'Mangiare', 'Lavare stoviglie', 'Pulizia casa',
    'Cucinare', 'Spostamenti', 'Attività fuori casa', 'Lavare bucato', 'Guardare TV/PC',
    'Sport', 'Contatti sociali', 'Salute', 'Lavoro', 'Lettura', 'Tempo libero a casa',
    'Uso PC', 'Ascoltare musica', 'Manutenzione domestica', 'Stiratura', 'Studio', 'Telefono', 'Altro'
]

activity_english_dict = {
    'Dormire/Riposo'            : 'Sleeping/Resting',
    'Igiene e vestirsi'         : 'Personal care & dressing',
    'Mangiare'                  : 'Eating',
    'Lavare stoviglie'          : 'Washing dishes',
    'Pulizia casa'              : 'House cleaning',
    'Cucinare'                  : 'Cooking',
    'Spostamenti'               : 'Travelling/Commuting',
    'Attività fuori casa'       : 'Activities outside home',
    'Lavare bucato'             : 'Doing laundry',
    'Guardare TV/PC'            : 'Watching TV/PC',
    'Sport'                     : 'Sports',
    'Contatti sociali'          : 'Social interactions',
    'Salute'                    : 'Health-related activities',
    'Lavoro'                    : 'Work',
    'Lettura'                   : 'Reading',
    'Tempo libero a casa'       : 'Leisure at home',
    'Uso PC'                    : 'Using PC',
    'Ascoltare musica'          : 'Listening to music',
    'Manutenzione domestica'    : 'Home maintenance',
    'Stiratura'                 : 'Ironing',
    'Studio'                    : 'Studying',
    'Telefono'                  : 'Phone use',
    'Altro'                     : 'Other'
}

activity_color_map = {
    # Rest / personal routines
    'Dormire/Riposo': '#1b2a6b',
    'Igiene e vestirsi': '#4c72b0',
    'Mangiare': '#86b6f6',

    # Domestic work
    'Lavare stoviglie': '#56b4e9',
    'Pulizia casa': '#b3d7e8',
    'Cucinare': '#009e73',
    'Lavare bucato': '#e69f00',
    'Stiratura': '#44aa99',
    'Manutenzione domestica': '#117733',

    # Mobility / outside home
    'Spostamenti': '#7fbc41',
    'Attività fuori casa': '#f0e442',

    # Leisure / media
    'Guardare TV/PC': '#d55e00',
    'Tempo libero a casa': '#dd3497',
    'Uso PC': '#f781bf',
    'Ascoltare musica': '#fbb4ae',
    'Lettura': '#b07cc6',
    'Telefono': '#2a9d8f',

    # Social / health / sport
    'Sport': '#cc6677',
    'Contatti sociali': '#882255',
    'Salute': '#aa4499',

    # Work / study
    'Lavoro': '#6a3d9a',
    'Studio': '#88ccee',

    # Residual class
    'Altro': '#999999'
}



aggregated_places = [
    'Casa propria', 'Casa propria, spazi aperti', 'Seconda casa', 'Casa di altre persone', 'Albergo e simili',
    'Luogo di lavoro', 'Luogo formazione', 'Luogo chiuso: pasti', 'Luogo chiuso: commissioni', 'Luogo all\'aperto',
    'Luogo chiuso: tempo libero', 'Luogo di culto', 'Luogo n/d', 'Spostamento: net-zero', 'Spostamento: moto',
    'Spostamento: auto', 'Spostamento: HDV', 'Spostamento: acqua', 'Spostamento: TP', 'Spostamento: aria',
    'Altro luogo', 'Spostamento: n/d'
]


place_english_dict = {
    'Casa propria': 'Own home',
    'Casa propria, spazi aperti': 'Own home, outdoor spaces',
    'Seconda casa': 'Second home',
    'Casa di altre persone': 'Other people’s home',
    'Albergo e simili': 'Hotel and similar',
    'Luogo di lavoro': 'Workplace',
    'Luogo formazione': 'Place of education/training',
    'Luogo chiuso: pasti': 'Indoor place: meals',
    'Luogo chiuso: commissioni': 'Indoor place: errands',
    'Luogo all\'aperto': 'Outdoor place',
    'Luogo chiuso: tempo libero': 'Indoor place: leisure',
    'Luogo di culto': 'Place of worship',
    'Luogo n/d': 'Place n/a',
    'Spostamento: net-zero': 'Travel: walking/biking',
    'Spostamento: moto': 'Travel: motorcycle',
    'Spostamento: auto': 'Travel: car',
    'Spostamento: HDV': 'Travel: heavy-duty vehicle',
    'Spostamento: acqua': 'Travel: water transport',
    'Spostamento: TP': 'Travel: public transport',
    'Spostamento: aria': 'Travel: air transport',
    'Altro luogo': 'Other place',
    'Spostamento: n/d': 'Travel: n/a'
}

place_color_map = {
    # Home-related places
    'Casa propria': '#1b2a6b',
    'Casa propria, spazi aperti': '#4c72b0',
    'Seconda casa': '#86b6f6',
    'Casa di altre persone': '#8dd3c7',

    # Accommodation / work / education
    'Albergo e simili': '#b3b3b3',
    'Luogo di lavoro': '#6a3d9a',
    'Luogo formazione': '#88ccee',

    # Indoor public/commercial places
    'Luogo chiuso: pasti': '#e69f00',
    'Luogo chiuso: commissioni': '#f0e442',
    'Luogo chiuso: tempo libero': '#d55e00',
    'Luogo di culto': '#cc6677',

    # Outdoor / undefined
    'Luogo all\'aperto': '#009e73',
    'Luogo n/d': '#999999',
    'Altro luogo': '#666666',

    # Travel modes
    'Spostamento: net-zero': '#7fbc41',
    'Spostamento: moto': '#44aa99',
    'Spostamento: auto': '#117733',
    'Spostamento: HDV': '#332288',
    'Spostamento: acqua': '#56b4e9',
    'Spostamento: TP': '#0072b2',
    'Spostamento: aria': '#882255',
    'Spostamento: n/d': '#bbbbbb'
}


# Dictionaries for periods of the year (days, quarters, individual classes)
# These values are intentionally kept in Italian because they are ISTAT source-data labels.
source_day_types = [
    'feriale',
    'sabato',
    'domenica'
    ]

day_english_dict = {
    'feriale': 'Working day',
    'sabato': 'Saturday',
    'domenica': 'Sunday'
    }

# These values are intentionally kept in Italian because the quarter abbreviations are matched to the ISTAT source-data labels.
source_quarters = [
    'Novembre, Dicembre, Gennaio',
    'Febbraio, Marzo, Aprile',
    'Maggio, Giugno, Luglio',
    'Agosto, Settembre, Ottobre'
    ]

quarter_abbr_english_dict = {
    'Nov-Dic-Gen': 'Nov-Dec-Jan',
    'Feb-Mar-Apr': 'Feb-Mar-Apr',
    'Mag-Giu-Lug': 'May-Jun-Jul',
    'Ago-Set-Ott': 'Aug-Sep-Oct'
    }

# English labels used only in matrix filenames.
MATRIX_DAY_FILE_LABELS = {
    "feriale": "working_day",
    "sabato": "saturday",
    "domenica": "sunday",
}

MATRIX_SEASON_FILE_LABELS = {
    "Nov-Dic-Gen": "winter",
    "Feb-Mar-Apr": "spring",
    "Mag-Giu-Lug": "summer",
    "Ago-Set-Ott": "autumn",
}

# Compact English prefixes used for the different matrix files.
MATRIX_FILE_STEMS = {
    "transition_matrix_activity": "activ_trans_mtx",
    "initial_prob_activity": "initial_prob_activity",
    "transition_matrix_places": "place_trans_mtx",
    "initial_prob_places": "initial_prob_place",
}




# Still to use Italian names if used ISTAT (i.e. Italian) data
quarter_abbreviation_map = {
    "Novembre, Dicembre, Gennaio": "Nov-Dic-Gen",
    "Febbraio, Marzo, Aprile": "Feb-Mar-Apr",
    "Maggio, Giugno, Luglio": "Mag-Giu-Lug",
    "Agosto, Settembre, Ottobre": "Ago-Set-Ott",
    "Nov-Dic-Gen": "Nov-Dic-Gen",
    "Feb-Mar-Apr": "Feb-Mar-Apr",
    "Mag-Giu-Lug": "Mag-Giu-Lug",
    "Ago-Set-Ott": "Ago-Set-Ott",
}


suffix_dict = {
    (0, 0): 'membro_isolato',
    (1, 1): 'coppiaCF_caponucleo' ,
    (1, 2): 'coppiaCF_coniuge',
    (1, 3): 'coppiaCF_figlio',
    (2, 1): 'coppiaSF_caponu',
    (2, 2): 'coppiaSF_coniug',
    (3, 1): 'Mono_M_padre',
    (3, 3): 'Mono_M_figlio',
    (4, 1): 'Mono_F_madre',
    (4, 3): 'Mono_F_figlio'
    }

suffix_english_dict = {
    (0, 0): 'single',
    (1, 1): 'couple_w_child_head' ,
    (1, 2): 'couple_w_child_spouse',
    (1, 3): 'couple_w_child_child',
    (2, 1): 'couple_wo_child_head',
    (2, 2): 'couple_wo_child_spouse',
    (3, 1): 'mono_M_father',
    (3, 3): 'mono_M_child',
    (4, 1): 'mono_F_mother',
    (4, 3): 'mono_F_child'
    }


italy_regions_dict = {
    '1': 'Nord-ovest',
    '2': 'Nord-est',
    '3': 'Centro',
    '4': 'Sud',
    '5': 'Isole'
}

region_abbr_dict = {
    'Nord-ovest': 'N-O',
    'Nord-est': 'N-E',
    'Centro': 'Cen',
    'Sud': 'Sud',
    'Isole': 'Isl'
}


quarter_abbr_dict = {
    'Novembre, Dicembre, Gennaio': 'Nov-Dic-Gen',
    'Febbraio, Marzo, Aprile': 'Feb-Mar-Apr',
    'Maggio, Giugno, Luglio': 'Mag-Giu-Lug',
    'Agosto, Settembre, Ottobre': 'Ago-Set-Ott'
}


# Names that identify pooled areas in matrix file names.
region_group_names = {"N-O_N-E_Cen", "Sud_Isl"}







# -------------------------------------------------------------------------
#   Sec 1.3.1 -- 2) PLOT DIMENSIONS
# -------------------------------------------------------------------------

PLOT_STYLE_PRESETS = {
    "single_column": {
        "num_ticks": 8,
        "fig_size": (3.55, 3.10),
        "font_size": 8,
        "label_font_size": 8.5,
        "tick_font_size": 7.5,
        "legend_font_size": 5.8,
        "legend_position": "bottom",
        "legend_ncol": 3,
    },
    "two_column": {
        "num_ticks": 12,
        "fig_size": (7.20, 3.70),
        "font_size": 9.5,
        "label_font_size": 10.5,
        "tick_font_size": 9,
        "legend_font_size": 7.2,
        "legend_position": "right",
        "legend_ncol": 1,
    },
}

### 1.3.2 Explicit hybrid geographical policy for transition matrices

The table `MATRIX_POLICY_BY_CONFIGURATION` is the single source of truth. It determines both:

1. which ISTAT regions are pooled when matrices are estimated in Section 1.8;
2. which matrix area each regional annual profile uses in Section 1.9.

The policy is therefore no longer implemented by manually commenting and uncommenting regional loops.


In [ ]:
# -------------------------------------------------------------------------
#   Sec 1.3 -- 2) MATRIX GEOGRAPHICAL POLICIES
# -------------------------------------------------------------------------
# Each policy lists the independent source-data pools used to estimate the
# transition matrices. Every ISTAT region code must occur exactly once in each
# policy. The same policy is later used to map each requested regional profile
# to the correct matrix folder.
MATRIX_AREA_POLICIES = {
    # Five distinct matrix sets: one for each ISTAT macro-region.
    "region_specific": (
        {"area_abbr": "N-O", "region_codes": ("1",)},
        {"area_abbr": "N-E", "region_codes": ("2",)},
        {"area_abbr": "Cen", "region_codes": ("3",)},
        {"area_abbr": "Sud", "region_codes": ("4",)},
        {"area_abbr": "Isl", "region_codes": ("5",)},
    ),

    # Two pooled matrix sets, used when samples are insufficient at the
    # individual-region level. Neighbouring geographical areas are considered for merging.
    "two_macro_regions": (
        {"area_abbr": "N-O_N-E_Cen", "region_codes": ("1", "2", "3")},
        {"area_abbr": "Sud_Isl", "region_codes": ("4", "5")},
    ),

    # North-West, North-East and Centre remain separate, while South and
    # Islands are pooled because their samples are insufficient separately.
    "south_islands_grouped": (
        {"area_abbr": "N-O", "region_codes": ("1",)},
        {"area_abbr": "N-E", "region_codes": ("2",)},
        {"area_abbr": "Cen", "region_codes": ("3",)},
        {"area_abbr": "Sud_Isl", "region_codes": ("4", "5")},
    ),
}

# Explicit policy selected after inspecting the number of Time-Use Survey
# respondents available for every household-type/member class. This table is the
# single source of truth for the hybrid geographical structure.
#
#   ------------------
#   Change this table if a different dataset or a different minimum-sample rule
#   is adopted. Keeping every supported configuration explicit prevents a new
#   class from silently inheriting an unsuitable geographical aggregation.
#   ------------------
MATRIX_POLICY_BY_CONFIGURATION = {
    (0, 0): "region_specific",          # single person
    (1, 1): "region_specific",          # couple with children: head
    (1, 2): "region_specific",          # couple with children: spouse
    (1, 3): "region_specific",          # couple with children: child
    (2, 1): "region_specific",          # couple without children: head
    (2, 2): "region_specific",          # couple without children: spouse
    (3, 1): "two_macro_regions",        # male single parent: father
    (3, 3): "two_macro_regions",        # male single parent: child
    (4, 1): "south_islands_grouped",    # female single parent: mother
    (4, 3): "south_islands_grouped",    # female single parent: child
}


# Map the English day_name to the Italian source/file label
source_day_map = {
    'Saturday':  'sabato',
    'Sunday':    'domenica'
    # all other days -> 'feriale' (legacy ISTAT/source label)
}


## 1.4 Simulation parameters

In [ ]:
# Household-type + household-position configurations to analyze (see description in Section 1.3 for all considered individual classes)
# Example: (1, 3) means couple with children — child.
# individual_configurations = [(0, 0)]
# to consider all individual classes uncomment:
individual_configurations = [
    (0, 0),
    (1, 1),
    (1, 2),
    (1, 3),
    (2, 1),
    (2, 2),
    (3, 1),
    (3, 3),
    (4, 1),
    (4, 3),
]

# Number of people for whom to create a synthetic profile
simulated_people = 200

# Time parameters (WARNING: if the selected periods are changed, the COOLING profiles and other downstream profiles must also be updated!, as PV generation, space heating/outdoor environmental parameters)
simulation_year = 2023
simulation_days = 366 if pd.Timestamp(f"{simulation_year}-01-01").is_leap_year else 365
num_time_steps = 144  # 10-minute resolution

# Weeks for which profiles are extracted
selected_periods = {
    'Winter profiles':      ('2023-02-15 00:00:00', '2023-02-21 23:50:00'),
    'Spring profiles':      ('2023-04-19 00:00:00', '2023-04-25 23:50:00'),
    'Summer profiles':      ('2023-07-12 00:00:00', '2023-07-18 23:50:00'),
    'Autumn profiles':      ('2023-10-11 00:00:00', '2023-10-17 23:50:00'),
    'Final winter day':     ('2023-12-20 00:00:00', '2023-12-20 23:50:00'),
}

## 1.5 Directories

In [ ]:
# Inputs
input_folder = Path("input") / "01_input_profiles_att"
legacy_input_folder = Path("input") / "01_input_profiles_att_OLD"
istat_original_tus_data_folder = input_folder / "ISTAT original TUS data"

# Outputs
output_folder = Path("output")
output_pictures_folder = output_folder / "01_profiles_Pics"
activity_matrices_folder = output_folder / "01_Mtx_activ"
location_matrices_folder = output_folder / "01_Mtx_places"
yearly_activity_profiles_folder = output_folder / "01_activity_profiles" / "yearly_profiles"
weekly_activity_profiles_folder = output_folder / "01_activity_profiles" / "weekly_profiles"
diagnostics_folder = output_folder / "01_diagnostics"

for folder in (
    input_folder,
    istat_original_tus_data_folder,
    output_pictures_folder,
    activity_matrices_folder,
    location_matrices_folder,
    yearly_activity_profiles_folder,
    weekly_activity_profiles_folder,
    diagnostics_folder,
):
    folder.mkdir(parents=True, exist_ok=True)

## 1.6 Function definitions

### 1.6.1 Input reconstruction, filters, matrix calculation, and plotting helpers

In [ ]:
# ----------------------------------------------------------
#       Sec 1.6.1 -- 1) FUNCTIONS FOR REBUILDING AGGREGATED DATASETS (starting from the original TUS data)
# ----------------------------------------------------------

# Function to save created picle compressed files
def save_pickle_gzip(obj, file_path):
    """Save a Python object as a gzip-compressed pickle file."""
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    with gzip.open(file_path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


# Function to load pickle files
def load_pickle_gzip(file_path):
    """Load a Python object from a gzip-compressed pickle file."""
    with gzip.open(file_path, "rb") as f:
        return pickle.load(f)


# Function to align original TUS data with activity aggregation defined by the authors
def normalize_mapping_code(series, zfill=None, strip_leading_zeros=False):
    """
    Normalize ISTAT code columns to make source-data codes and legend-table codes comparable.
    """
    normalized_series = (
        series.astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

    if strip_leading_zeros:
        # Preserve the valid code 0: lstrip("0") would otherwise turn it into
        # an empty string and break mappings for single-person households.
        normalized_series = normalized_series.str.lstrip("0").replace("", "0")

    if zfill is not None:
        normalized_series = normalized_series.str.zfill(zfill)

    return normalized_series



# Function to adjust nomenclature for quarters in the file
def convert_quarter_labels_to_abbreviations(df):
    """
    Convert the original Italian quarter labels into the abbreviated labels used in this script.

    The internal/source-facing labels remain the Italian abbreviations:
        - Nov-Dic-Gen
        - Feb-Mar-Apr
        - Mag-Giu-Lug
        - Ago-Set-Ott
    """

    if "Trimestre" in df.columns:
        df["Trimestre"] = (
            df["Trimestre"]
            .astype(str)
            .str.strip()
            .map(quarter_abbreviation_map)
            .fillna(df["Trimestre"].astype(str).str.strip())
        )

    return df


# Function to rebuild "df_individui_full" dataframe with all information for individuals
def build_individuals_dataframe(input_dir):
    """
    Build the full individuals dataframe from the original ISTAT individuals file.

    Output:
        df_individui_full.pkl.gz
    """
    input_dir = Path(input_dir)

    individuals_file_path = istat_original_tus_data_folder / "uso_tempo_Microdati_Anno_2013_Individui.txt"              # original ISTAT TUS dataset (to be adjusted if used EUROSTAT one)
    legend_file_path = input_dir / "descrizione_dati_TUS.xlsx"                                                          # aggregation list for activities, places, individuals defined by the authors

    individuals_df = pd.read_csv(individuals_file_path, delimiter="\t", dtype=str)

    individuals_legend_df = pd.read_excel(legend_file_path, sheet_name="Legenda")

    # category definition (see the original ISTAT TUS data documentation for the meaning of each class)
    code_columns = [
        "rk3", "tipnu2", "rpnuc2", "tipfa2m", "tipoab",
        "distsc", "abitlo", "auto", "bic", "motor", "amoto"
    ]

    for column in code_columns:
        if column in individuals_df.columns:
            individuals_df[column] = individuals_df[column].astype(str).str.strip()

        if column in individuals_legend_df.columns:
            individuals_legend_df[column] = normalize_mapping_code(
                individuals_legend_df[column],
                strip_leading_zeros=True
            )

    # Description/mapping of each TUS column
    diary_mapping = dict(zip(
        individuals_legend_df["rk3"],
        individuals_legend_df["Diario giornaliero"]
    ))

    household_type_mapping = dict(zip(
        individuals_legend_df["tipnu2"],
        individuals_legend_df["tipo di nucleo"]
    ))

    household_position_mapping = dict(zip(
        individuals_legend_df["rpnuc2"],
        individuals_legend_df["posizione nel nucleo familiare"]
    ))

    dwelling_type_mapping = dict(zip(
        individuals_legend_df["tipoab"],
        individuals_legend_df["tipologia abitazione"]
    ))

    study_distance_mapping = dict(zip(
        individuals_legend_df["distsc"],
        individuals_legend_df["Lontananza della sua abitazione dal luogo dove si svolge il corso"]
    ))

    work_distance_mapping = dict(zip(
        individuals_legend_df["abitlo"],
        individuals_legend_df["Distanza abitazione da luogo di lavoro"]
    ))

    car_ownership_mapping = dict(zip(
        individuals_legend_df["auto"],
        individuals_legend_df["la famiglia possiede: automobile"]
    ))

    bike_ownership_mapping = dict(zip(
        individuals_legend_df["bic"],
        individuals_legend_df["la famiglia possiede: bicicletta"]
    ))

    scooter_ownership_mapping = dict(zip(
        individuals_legend_df["motor"],
        individuals_legend_df["la famiglia possiede: motorino, scooter"]
    ))

    motorcycle_ownership_mapping = dict(zip(
        individuals_legend_df["amoto"],
        individuals_legend_df["la famiglia possiede: motocicletta, moto"]
    ))

    individuals_df["Diario giornaliero"] = individuals_df["rk3"].map(diary_mapping)
    individuals_df["tipo di nucleo"] = individuals_df["tipnu2"].map(household_type_mapping)
    individuals_df["posizione nel nucleo familiare"] = individuals_df["rpnuc2"].map(household_position_mapping)
    individuals_df["tipologia abitazione"] = individuals_df["tipoab"].map(dwelling_type_mapping)
    individuals_df["Lontananza della sua abitazione dal luogo dove si svolge il corso"] = individuals_df["distsc"].map(study_distance_mapping)
    individuals_df["Distanza abitazione da luogo di lavoro"] = individuals_df["abitlo"].map(work_distance_mapping)
    individuals_df["la famiglia possiede: automobile"] = individuals_df["auto"].map(car_ownership_mapping)
    individuals_df["la famiglia possiede: bicicletta"] = individuals_df["bic"].map(bike_ownership_mapping)
    individuals_df["la famiglia possiede: motorino, scooter"] = individuals_df["motor"].map(scooter_ownership_mapping)
    individuals_df["la famiglia possiede: motocicletta, moto"] = individuals_df["amoto"].map(motorcycle_ownership_mapping)

    # Unique person identifier used throughout the following dataframes.
    individuals_df["persona"] = (
        individuals_df["profam"].astype(str) + "." + individuals_df["proind"].astype(str)
    )

    # Move the person identifier to the first column.
    columns = individuals_df.columns.tolist()
    columns.insert(0, columns.pop(columns.index("persona")))
    individuals_df = individuals_df[columns]

    save_pickle_gzip(
        individuals_df,
        input_dir / "df_individui_full.pkl.gz"
    )

    return individuals_df



# Function to properly split activities that cross midnight (e.g., sleeping from 23:30 to 06:30)
def split_rows_crossing_midnight(df, start_column="ora inizio", end_column="ora fine"):
    """
    Split rows whose start time is later than their end time, meaning that
    the recorded activity/place crosses midnight.

    Example:
        23:30 -> 06:30

    becomes:
        23:30 -> 23:59
        00:00 -> 06:30
    """
    df = df.copy()

    df[start_column] = pd.to_datetime(
        df[start_column],
        format="%H:%M",
        errors="coerce"
    )

    df[end_column] = pd.to_datetime(
        df[end_column],
        format="%H:%M",
        errors="coerce"
    )

    crosses_midnight_mask = df[start_column] > df[end_column]

    if not crosses_midnight_mask.any():
        return df

    before_midnight_df = df.loc[crosses_midnight_mask].copy()
    after_midnight_df = df.loc[crosses_midnight_mask].copy()

    before_midnight_df[end_column] = pd.to_datetime("23:59", format="%H:%M")
    after_midnight_df[start_column] = pd.to_datetime("00:00", format="%H:%M")

    df = pd.concat(
        [
            df.loc[~crosses_midnight_mask],
            before_midnight_df,
            after_midnight_df
        ],
        ignore_index=True
    )

    return df



# Function to rebuild the full daily activity dataframe from ISTAT data
def build_daily_diary_dataframe(input_dir):
    """
    Build the full daily diary dataframe from the original ISTAT daily diary file.

    Output:
        df_diario_giornaliero_full.pkl.gz
    """
    input_dir = Path(input_dir)

    diary_file_path  = istat_original_tus_data_folder / "uso_tempo_Microdati_Anno_2013_DiarioGiornaliero.txt"               # original ISTAT TUS dataset (to be adjusted if used EUROSTAT one)
    legend_file_path = input_dir / "descrizione_dati_TUS.xlsx"                                                              # aggregation list for activities, places, individuals defined by the authors

    diary_df = pd.read_csv(
        diary_file_path,
        delimiter="\t",
        dtype=str
    )

    diary_legend_df = pd.read_excel(
        legend_file_path,
        sheet_name="Legenda"
    )

    # Normalize source-data codes.
    diary_df["cluogo"] = normalize_mapping_code(diary_df["cluogo"], zfill=3)
    diary_df["catpri"] = normalize_mapping_code(diary_df["catpri"], zfill=3)
    diary_df["meseri"] = diary_df["meseri"].astype(str).str.strip()
    diary_df["catcon"] = diary_df["catcon"].astype(str).str.strip()
    diary_df["gsett"] = diary_df["gsett"].astype(str).str.strip()

    # Normalize legend-table codes.
    diary_legend_df["cluogo"] = normalize_mapping_code(diary_legend_df["cluogo"], zfill=3)
    diary_legend_df["catpri"] = normalize_mapping_code(diary_legend_df["catpri"], zfill=3)
    diary_legend_df["meseri"] = normalize_mapping_code(diary_legend_df["meseri"])
    diary_legend_df["catcon"] = normalize_mapping_code(diary_legend_df["catcon"])
    diary_legend_df["gsett"] = normalize_mapping_code(diary_legend_df["gsett"])

    place_mapping = dict(zip(
        diary_legend_df["cluogo"],
        diary_legend_df["Luogo aggregato"]
    ))

    activity_mapping = dict(zip(
        diary_legend_df["catpri"],
        diary_legend_df["Attività aggregata"]
    ))

    quarter_mapping = dict(zip(
        diary_legend_df["meseri"],
        diary_legend_df["Trimestre"]
    ))

    condition_mapping = dict(zip(
        diary_legend_df["catcon"],
        diary_legend_df["Descrizione catcon"]
    ))

    day_type_mapping = dict(zip(
        diary_legend_df["gsett"],
        diary_legend_df["Descrizione gsett"]
    ))

    diary_df["Luogo aggregato"] = diary_df["cluogo"].map(place_mapping)
    diary_df["Attività aggregata"] = diary_df["catpri"].map(activity_mapping)
    diary_df["Trimestre"] = diary_df["meseri"].map(quarter_mapping)
    diary_df["Descrizione catcon"] = diary_df["catcon"].map(condition_mapping)
    diary_df["Descrizione gsett"] = diary_df["gsett"].map(day_type_mapping)


    # Align quarter labels with the rest of this script.
    diary_df = convert_quarter_labels_to_abbreviations(diary_df)

    # Align the place label with the list used in ENG_01.
    diary_df["Luogo aggregato"] = diary_df["Luogo aggregato"].replace({
        "Luogo in esterno": "Luogo all'aperto"
    })

    diary_df["ora inizio"] = (
        diary_df["oraini"].str.zfill(2) + ":" + diary_df["minini"].str.zfill(2)
    )

    diary_df["ora fine"] = (
        diary_df["orafin"].str.zfill(2) + ":" + diary_df["minfin"].str.zfill(2)
    )

    diary_df["persona"] = (
        diary_df["profam"].astype(str) + "." + diary_df["proind"].astype(str)
    )

    # Align with the legacy pipeline:
    # the daily diary dataframe is saved after splitting rows that cross midnight.
    diary_df = split_rows_crossing_midnight(
        diary_df,
        start_column="ora inizio",
        end_column="ora fine"
    )

    save_pickle_gzip(
        diary_df,
        input_dir / "df_diario_giornaliero_full.pkl.gz"
    )

    return diary_df



# === Main function to create dataframe with 144 ts for aggregated activities and aggregated places
def build_144_timestep_dataframe(
        diary_df,
        category_column,
        categories,
        output_file_path,
        progress_description):
    """
    Build a person-by-timestep dataframe with 144 ten-minute timesteps.

    The same function is used for:
        - aggregated activities
        - aggregated occupied places

    Output format:
        persona | time_step_0 | time_step_1 | ... | time_step_143
    """
    working_df = diary_df.copy()

    category_to_code = {
        category: i + 1
        for i, category in enumerate(categories)
    }

    working_df["_start_time"] = pd.to_datetime(
        working_df["ora inizio"],
        errors="coerce"
    )

    working_df["_end_time"] = pd.to_datetime(
        working_df["ora fine"],
        errors="coerce"
    )

    # Split activities or places that cross midnight into two rows.
    crosses_midnight_mask = working_df["_start_time"] > working_df["_end_time"]

    if crosses_midnight_mask.any():
        before_midnight_df = working_df.loc[crosses_midnight_mask].copy()
        after_midnight_df = working_df.loc[crosses_midnight_mask].copy()

        before_midnight_df["_end_time"] = pd.to_datetime("23:59", format="%H:%M")
        after_midnight_df["_start_time"] = pd.to_datetime("00:00", format="%H:%M")

        working_df = pd.concat(
            [
                working_df.loc[~crosses_midnight_mask],
                before_midnight_df,
                after_midnight_df
            ],
            ignore_index=True
        )

    time_intervals = pd.date_range(
        start="00:00",
        end="23:50",
        freq="10min"
    )

    time_step_columns = [
        f"time_step_{i}"
        for i in range(len(time_intervals))
    ]

    people = working_df["persona"].drop_duplicates().tolist()

    person_timestep_dict = {
        person: [0] * len(time_intervals)
        for person in people
    }

    for person, person_df in tqdm(
            working_df.groupby("persona", sort=False),
            desc=progress_description):

        for time_step_index, time_interval in enumerate(time_intervals):
            active_rows = person_df[
                (person_df["_start_time"].dt.time <= time_interval.time()) &
                (person_df["_end_time"].dt.time > time_interval.time())
            ]

            if not active_rows.empty:
                category_value = active_rows.iloc[0][category_column]
                person_timestep_dict[person][time_step_index] = category_to_code.get(
                    category_value,
                    0
                )

    output_data = [
        [person] + time_step_values
        for person, time_step_values in person_timestep_dict.items()
    ]

    timestep_df = pd.DataFrame(
        output_data,
        columns=["persona"] + time_step_columns
    )

    timestep_df[time_step_columns] = timestep_df[time_step_columns].astype(int)

    # ---- debug check
    missing_values_count = (timestep_df[time_step_columns] == 0).sum().sum()

    if missing_values_count > 0:
        print(f"WARNING: {missing_values_count} timestep cells have value 0 because no category was mapped.")
    # ----------------

    save_pickle_gzip(
        timestep_df,
        output_file_path
    )

    return timestep_df



# Function to build all input dataframes
def rebuild_input_dataframes(input_dir):
    """Rebuild and save all source-derived input DataFrames.

    Large intermediate DataFrames are released as soon as they are no longer
    required. The function returns only shape metadata, not the DataFrames.
    """
    input_dir = Path(input_dir)
    input_dir.mkdir(parents=True, exist_ok=True)

    print("-" * 60)
    print("Rebuilding MODULE_01 input DataFrames from ISTAT source files")
    print("-" * 60)

    print("1/5 - Building full individuals DataFrame...")
    individuals_df = build_individuals_dataframe(input_dir)
    individuals_shape = individuals_df.shape

    print("2/5 - Building full daily diary DataFrame...")
    daily_diary_df = build_daily_diary_dataframe(input_dir)
    daily_diary_shape = daily_diary_df.shape

    print("3/5 - Building 144-timestep activity DataFrame...")
    activity_df = build_144_timestep_dataframe(
        diary_df=daily_diary_df,
        category_column="Attività aggregata",
        categories=aggregated_activities,
        output_file_path=input_dir / "df_144timestep_ISTAT_full.pkl.gz",
        progress_description="Building activity 144-timestep DataFrame",
    )
    activity_shape = activity_df.shape
    del activity_df
    gc.collect()

    print("4/5 - Building 144-timestep occupied-place DataFrame...")
    place_df = build_144_timestep_dataframe(
        diary_df=daily_diary_df,
        category_column="Luogo aggregato",
        categories=aggregated_places,
        output_file_path=input_dir / "df_luoghi_144_timestep_ISTAT_full.pkl.gz",
        progress_description="Building occupied-place 144-timestep DataFrame",
    )
    place_shape = place_df.shape
    del place_df
    gc.collect()

    print("5/5 - Merging daily diary and individuals DataFrames...")
    diary_with_individuals_df = pd.merge(
        daily_diary_df,
        individuals_df,
        on="persona",
        how="inner",
    )
    merged_shape = diary_with_individuals_df.shape
    save_pickle_gzip(
        diary_with_individuals_df,
        input_dir / "df_diario_giorn_con_individui_full.pkl.gz",
    )

    del diary_with_individuals_df, daily_diary_df, individuals_df
    gc.collect()

    shape_summary = {
        "individuals": individuals_shape,
        "daily_diary": daily_diary_shape,
        "diary_with_individuals": merged_shape,
        "activity_144_timestep": activity_shape,
        "place_144_timestep": place_shape,
    }

    print("-" * 60)
    print("Input DataFrame rebuild completed")
    for name, shape in shape_summary.items():
        print(f"{name}: {shape}")
    print("-" * 60)
    return shape_summary


INPUT_DATA_PATHS = {
    "diary": input_folder / "df_diario_giorn_con_individui_full.pkl.gz",
    "activity": input_folder / "df_144timestep_ISTAT_full.pkl.gz",
    "place": input_folder / "df_luoghi_144_timestep_ISTAT_full.pkl.gz",
}



# Function to load TUS description table from excel
def load_description_dictionary():
    """Load the TUS description table only when a profile-output stage needs it."""
    description_path = input_folder / "descrizione_dati_TUS.xlsx"
    if not description_path.exists():
        raise FileNotFoundError(f"Description file not found: {description_path.resolve()}")
    return pd.read_excel(description_path)


# Function to load the large required inputs (i.e., TUS diary, activity or occupied places datasets in .pkl; designed to load only the required inputs, instead of loading all of them always)
def load_required_inputs(*, need_diary=False, need_activity=False, need_place=False):
    """Load only the large input DataFrames required by the selected stage."""
    requested = {
        "diary": need_diary,
        "activity": need_activity,
        "place": need_place,
    }
    requested_paths = {
        name: INPUT_DATA_PATHS[name]
        for name, is_required in requested.items()
        if is_required
    }

    missing_paths = [path for path in requested_paths.values() if not path.exists()]
    if missing_paths:
        if AUTO_REBUILD_MISSING_INPUTS:
            print("Required inputs are missing; automatic rebuild explicitly enabled.")
            rebuild_input_dataframes(input_folder)
            missing_paths = [path for path in requested_paths.values() if not path.exists()]
        if missing_paths:
            missing_text = "\n".join(f" - {path.resolve()}" for path in missing_paths)
            raise FileNotFoundError(
                "Required MODULE_01 input files are missing:\n"
                f"{missing_text}\n"
                "Check the working directory, or run only RUN_AGGR_DATASETS=True. "
                "Automatic rebuilding is disabled by default for memory safety."
            )

    loaded = {}
    for name, path in requested_paths.items():
        print(f"Loading {name} input: {path}")
        loaded[name] = load_pickle_gzip(path)
        print(f"Loaded {name}: shape={loaded[name].shape}")
    return loaded


# -------------------------------------------------------------------------------








# ----------------------------------------------------------
#       Sec 1.6.1 -- 2) FUNCTIONS FOR LOADING, AGGREGATION AND FILTERS
# ----------------------------------------------------------

## FUNCTION FOR APPLYING FILTERS TO THE DATAFRAME

def apply_filters(
        diary_df,
        day_type,
        quarter,
        regions,
        household_types,
        household_positions):
    """Filter the diary using the operational ISTAT dimensions used by MODULE_01."""
    normalized_household_types = [str(value).zfill(2) for value in household_types]
    normalized_household_positions = [str(value).zfill(2) for value in household_positions]

    return diary_df[
        (diary_df["Descrizione gsett"] == day_type)
        & (diary_df["Trimestre"] == quarter)
        & diary_df["rip"].astype(str).isin([str(value) for value in regions])
        & diary_df["tipnu2"].astype(str).str.zfill(2).isin(normalized_household_types)
        & diary_df["rpnuc2"].astype(str).str.zfill(2).isin(normalized_household_positions)
    ]




## FUNCTION FOR CREATING THE AGGREGATED DATAFRAME (ACTIVITIES AND PLACES) USING VECTORIZATION
def create_aggregated_df_vectorized(df, categories):
    n_timesteps = 144
    n_cat = len(categories)
    # Build the list of columns where data for each timestep is stored.
    time_cols = [f'time_step_{t}' for t in range(n_timesteps)]


    # Extract data from the specified columns into a NumPy array.
    # Values are assumed to be integers from 1 to n_cat.
    data = df[time_cols].values.astype(int)

    # Create an empty matrix for counts: (n_timesteps, n_cat)
    agg = np.zeros((n_timesteps, n_cat), dtype=int)

    for t in range(n_timesteps):
        # Extract the values at timestep t for all individuals
        values = data[:, t]
        # Filter out any invalid values (as a safety check, e.g., if the value is < 1)
        valid_values = values[values >= 1]

        # Calculate counts using np.bincount.
        # Since values range from 1 to n_cat, set minlength to n_cat+1
        # and then discard the count at position 0.
        if valid_values.size > 0:
            counts = np.bincount(valid_values, minlength=n_cat+1)[1:]
        else:
            counts = np.zeros(n_cat, dtype=int)
        agg[t, :] = counts

    # Create a DataFrame for aggregated counts, where each row corresponds to a timestep
    aggregated_df = pd.DataFrame(agg, columns=categories)
    return aggregated_df
# -------------------------------------------------------------------



# ----------------------------------------------------------
# Sec 1.6.1 -- 3) Region policy, transition matrices, simulation 
# ----------------------------------------------------------

# Function to return the policy name linked for geographical area associated to each individual class (eg male with children --> linked to XX geographical group; single person --> linked to YY single region)
def get_matrix_policy_name(tipnu2_value, rpnuc2_value):
    """Return the geographical matrix policy for one individual class.

    `MATRIX_POLICY_OVERRIDE` can temporarily force one policy for all classes.
    With the default value ``None``, the explicit configuration table is used.
    """
    configuration = (int(tipnu2_value), int(rpnuc2_value))

    if MATRIX_POLICY_OVERRIDE is not None:
        policy_name = MATRIX_POLICY_OVERRIDE
    else:
        try:
            policy_name = MATRIX_POLICY_BY_CONFIGURATION[configuration]
        except KeyError as error:
            raise KeyError(
                "No geographical matrix policy is defined for individual "
                f"configuration {configuration}. Add it to "
                "MATRIX_POLICY_BY_CONFIGURATION."
            ) from error

    if policy_name not in MATRIX_AREA_POLICIES:
        raise ValueError(
            f"Unknown matrix policy '{policy_name}' for configuration "
            f"{configuration}. Supported policies: "
            f"{sorted(MATRIX_AREA_POLICIES)}."
        )

    return policy_name


# Function to return the list of regions/geographical groups linked to each individual class
def get_matrix_area_cases(tipnu2_value, rpnuc2_value):
    """Return the source-region pools used for one individual class.

    Each returned item contains:
        - ``area_abbr``: folder/file label of the estimated matrix set;
        - ``region_codes``: ISTAT regions pooled to estimate that matrix set.

    This function is used both when matrices are built and when annual profiles
    are generated, so the two stages cannot apply different geographical rules.
    """
    policy_name = get_matrix_policy_name(tipnu2_value, rpnuc2_value)

    # Return mutable copies so callers cannot alter the central policy table.
    return [
        {
            "area_abbr": area_case["area_abbr"],
            "region_codes": list(area_case["region_codes"]),
        }
        for area_case in MATRIX_AREA_POLICIES[policy_name]
    ]


# Function to return, for each single region, the corresponding transition matrix to be used for the selected individual class (e.g., in N-O, for single person, used matrix calculated for N-O; for male with children in N-O, used matrix N-O_N-E_Cen)
def resolve_matrix_area(region_code, tipnu2_value, rpnuc2_value):
    """Return the matrix area supplying a requested regional profile."""
    region_code = str(region_code)

    for area_case in get_matrix_area_cases(tipnu2_value, rpnuc2_value):
        if region_code in area_case["region_codes"]:
            return area_case["area_abbr"]

    raise KeyError(
        f"No matrix area found for region={region_code}, "
        f"householdtype={tipnu2_value}, member={rpnuc2_value}."
    )


# Function to check each individual class has its own assignement of geographic derived matrix 
def validate_matrix_geographical_policies():
    """Validate and print the configuration-to-policy mapping.

    The validation ensures that:
        - every supported individual class has an explicit policy;
        - every referenced policy exists;
        - each policy covers region codes 1--5 exactly once;
        - every currently selected configuration is supported.
    """
    supported_configurations = set(suffix_english_dict)
    configured_classes = set(MATRIX_POLICY_BY_CONFIGURATION)

    missing_classes = supported_configurations - configured_classes
    extra_classes = configured_classes - supported_configurations

    if missing_classes or extra_classes:
        raise ValueError(
            "Invalid MATRIX_POLICY_BY_CONFIGURATION. "
            f"Missing classes: {sorted(missing_classes)}; "
            f"unknown classes: {sorted(extra_classes)}."
        )

    expected_region_codes = set(italy_regions_dict)

    for policy_name, area_cases in MATRIX_AREA_POLICIES.items():
        flattened_codes = [
            str(region_code)
            for area_case in area_cases
            for region_code in area_case["region_codes"]
        ]
        actual_codes = set(flattened_codes)
        duplicate_codes = sorted({
            code for code in flattened_codes
            if flattened_codes.count(code) > 1
        })

        if actual_codes != expected_region_codes or duplicate_codes:
            raise ValueError(
                f"Policy '{policy_name}' must cover ISTAT region codes "
                f"{sorted(expected_region_codes)} exactly once. "
                f"Actual codes: {flattened_codes}; duplicates: "
                f"{duplicate_codes}."
            )

        area_names = [area_case["area_abbr"] for area_case in area_cases]
        if len(area_names) != len(set(area_names)):
            raise ValueError(
                f"Policy '{policy_name}' contains duplicated area names: "
                f"{area_names}."
            )

    for configuration, policy_name in MATRIX_POLICY_BY_CONFIGURATION.items():
        if policy_name not in MATRIX_AREA_POLICIES:
            raise ValueError(
                f"Configuration {configuration} refers to unknown policy "
                f"'{policy_name}'."
            )

    unknown_selected = set(individual_configurations) - supported_configurations
    if unknown_selected:
        raise ValueError(
            "Unsupported individual configurations selected: "
            f"{sorted(unknown_selected)}."
        )

    print("Geographical matrix policy by individual configuration:")
    for configuration in sorted(MATRIX_POLICY_BY_CONFIGURATION):
        policy_name = get_matrix_policy_name(*configuration)
        area_description = "; ".join(
            f"{case['area_abbr']} <- {','.join(case['region_codes'])}"
            for case in get_matrix_area_cases(*configuration)
        )
        override_note = " [GLOBAL OVERRIDE]" if MATRIX_POLICY_OVERRIDE else ""
        print(
            f"  {configuration}: {policy_name}{override_note} | "
            f"{area_description}"
        )


# Function to insert, in the name file, "group" or "region" depending if it is used a grouped-region derived or a single region matrix
def area_file_prefix(area_abbr):
    """Return 'group' for macro-regions and 'region' for single regions."""
    return "group" if area_abbr in region_group_names else "region"


# Function to create the folder dinamically
def configuration_folder(base_dir, area_abbr, tipnu2_value, rpnuc2_value):
    """Create and return the folder for one area/individual configuration."""
    folder = (
        Path(base_dir)
        / area_abbr
        / f"householdtype_{tipnu2_value}_member_{rpnuc2_value}"
    )
    folder.mkdir(parents=True, exist_ok=True)
    return folder


# Function to build the compact English filename used for transition matrices and initial probabilities.
def build_matrix_file_name(
        matrix_kind,
        area_abbr,
        day_type,
        quarter_abbr,
        tipnu2_value,
        rpnuc2_value):

    if matrix_kind not in MATRIX_FILE_STEMS:
        raise ValueError(
            f"Unsupported matrix kind '{matrix_kind}'. "
            f"Choose from {tuple(MATRIX_FILE_STEMS)}."
        )

    if day_type not in MATRIX_DAY_FILE_LABELS:
        raise ValueError(
            f"Unsupported day type '{day_type}'. "
            f"Choose from {tuple(MATRIX_DAY_FILE_LABELS)}."
        )

    if quarter_abbr not in MATRIX_SEASON_FILE_LABELS:
        raise ValueError(
            f"Unsupported quarter '{quarter_abbr}'. "
            f"Choose from {tuple(MATRIX_SEASON_FILE_LABELS)}."
        )

    matrix_stem = MATRIX_FILE_STEMS[matrix_kind]
    day_label = MATRIX_DAY_FILE_LABELS[day_type]
    season_label = MATRIX_SEASON_FILE_LABELS[quarter_abbr]

    return (
        f"{matrix_stem}_{area_abbr}_{day_label}_{season_label}_"
        f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
    )


# Function to calculate transition matrices and initial probabilities using Markov Chain method
def calculate_transition_matrices_and_initial_probabilities(
        df,
        categories,
        assigned_region_name,
        week_day_type,
        quarter_abbr,
        num_time_steps=144):
    """
    Calculate the time-dependent transition probabilities.

    For each timestep t and pair of states i -> j:
        P(j | i, t) = count(i at t and j at t+1) / count(i at t)

    Rows with no observations are replaced by a uniform distribution.
    """
    if df.empty:
        raise ValueError(
            "Cannot calculate transition matrices from an empty dataframe: "
            f"area={assigned_region_name}, day={week_day_type}, quarter={quarter_abbr}."
        )

    num_categories = len(categories)
    transition_matrix = np.zeros(
        (num_time_steps, num_categories, num_categories),
        dtype=float,
    )
    initial_probabilities = np.zeros((num_categories, 1), dtype=float)

    for state_code in range(1, num_categories + 1):
        initial_probabilities[state_code - 1, 0] = (
            df["time_step_0"].eq(state_code).sum() / len(df)
        )

    initial_sum = np.nansum(initial_probabilities)
    if initial_sum > 0:
        initial_probabilities /= initial_sum
    else:
        initial_probabilities.fill(1 / num_categories)

    description = (
        "Calculating transition matrices "
        f"({assigned_region_name}, {week_day_type}, {quarter_abbr})"
    )

    for timestep in tqdm(range(num_time_steps - 1), desc=description):
        current_column = f"time_step_{timestep}"
        next_column = f"time_step_{timestep + 1}"

        for current_state in range(1, num_categories + 1):
            current_mask = df[current_column].eq(current_state)
            current_count = int(current_mask.sum())

            if current_count == 0:
                transition_matrix[timestep, current_state - 1, :] = 1 / num_categories
                continue

            for next_state in range(1, num_categories + 1):
                transition_count = int(
                    (current_mask & df[next_column].eq(next_state)).sum()
                )
                transition_matrix[
                    timestep,
                    current_state - 1,
                    next_state - 1,
                ] = transition_count / current_count

            row_sum = np.nansum(
                transition_matrix[timestep, current_state - 1, :]
            )
            if row_sum > 0:
                transition_matrix[timestep, current_state - 1, :] /= row_sum
            else:
                transition_matrix[timestep, current_state - 1, :] = 1 / num_categories

            transition_matrix[timestep, current_state - 1, :] = np.nan_to_num(
                transition_matrix[timestep, current_state - 1, :]
            )

    # Keep the 144-layer saved shape used by the original pipeline.
    transition_matrix[143] = transition_matrix[142]
    return transition_matrix, initial_probabilities


# Function to link month number with the corresponding defined quarter of the year 
def get_quarter(month):
    """Return the source-facing quarter abbreviation for a month number."""
    if month in (11, 12, 1):
        return "Nov-Dic-Gen"
    if month in (2, 3, 4):
        return "Feb-Mar-Apr"
    if month in (5, 6, 7):
        return "Mag-Giu-Lug"
    if month in (8, 9, 10):
        return "Ago-Set-Ott"
    raise ValueError(f"Invalid month: {month}")



# Function to use the correct activity matrix for the corresponding individual class
@lru_cache(maxsize=None)        # cache to avoid loading the same file many times during the annual activity profile generation process
def load_transition_cached(
        day_name,
        quarter,
        tipnu2_value,
        rpnuc2_value,
        area_abbr,
        path_base):

    path_base = Path(path_base)

    # Convert the Python day name to the label used in the source TUS data.
    source_day_type = source_day_map.get(day_name, "feriale")

    # Prefixes required only for compatibility with older matrix names.
    previous_english_prefix = (
        f"{area_file_prefix(area_abbr)}_{area_abbr}_"
    )
    legacy_italian_prefix = (
        f"grup_{area_abbr}_"
        if area_abbr in region_group_names
        else f"reg_{area_abbr}_"
    )

    matrix_file_candidates = [
        # New compact English convention.
        build_matrix_file_name(
            "transition_matrix_activity",
            area_abbr,
            source_day_type,
            quarter,
            tipnu2_value,
            rpnuc2_value,
        ),

        # Previous v05 English convention.
        (
            f"transition_matrix_activity_{previous_english_prefix}"
            f"{source_day_type}_{quarter}_"
            f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
        ),

        # Hybrid legacy convention.
        (
            f"mtx_trans_attiv_{legacy_italian_prefix}"
            f"{source_day_type}_{quarter}_"
            f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
        ),

        # Original Italian convention.
        (
            f"mtx_trans_attiv_{legacy_italian_prefix}"
            f"{source_day_type}_{quarter}_"
            f"tipnuc_{tipnu2_value}_membro_{rpnuc2_value}.npy"
        ),
    ]

    probability_file_candidates = [
        # New compact English convention.
        build_matrix_file_name(
            "initial_prob_activity",
            area_abbr,
            source_day_type,
            quarter,
            tipnu2_value,
            rpnuc2_value,
        ),

        # Previous v05 English convention.
        (
            f"initial_prob_activity_{previous_english_prefix}"
            f"{source_day_type}_{quarter}_"
            f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
        ),

        # Hybrid legacy convention.
        (
            f"prob_iniz_attiv_{legacy_italian_prefix}"
            f"{source_day_type}_{quarter}_"
            f"householdtype_{tipnu2_value}_member_{rpnuc2_value}.npy"
        ),

        # Original Italian convention.
        (
            f"prob_iniz_attiv_{legacy_italian_prefix}"
            f"{source_day_type}_{quarter}_"
            f"tipnuc_{tipnu2_value}_membro_{rpnuc2_value}.npy"
        ),
    ]

    def find_existing_file(candidate_names, file_kind):
        search_dirs = [
            path_base,
            path_base.parent,
            activity_matrices_folder,
        ]

        # Search first in the expected folders.
        for search_dir in search_dirs:
            for candidate_name in candidate_names:
                candidate_path = Path(search_dir) / candidate_name

                if candidate_path.exists():
                    return candidate_path

        # Recursive fallback for legacy folder structures.
        for candidate_name in candidate_names:
            matches = list(
                activity_matrices_folder.rglob(candidate_name)
            )

            if matches:
                return matches[0]

        checked_names = "\n".join(
            f" - {candidate_name}"
            for candidate_name in candidate_names
        )

        raise FileNotFoundError(
            f"{file_kind} not found for:\n"
            f"  day={source_day_type}\n"
            f"  quarter={quarter}\n"
            f"  householdtype={tipnu2_value}\n"
            f"  member={rpnuc2_value}\n"
            f"  matrix area={area_abbr}\n"
            f"  expected folder={path_base}\n\n"
            f"Candidate names:\n{checked_names}"
        )

    matrix_path = find_existing_file(
        matrix_file_candidates,
        "Activity transition matrix",
    )

    probability_path = find_existing_file(
        probability_file_candidates,
        "Activity initial probabilities",
    )

    return (
        np.load(matrix_path),
        np.load(probability_path),
    )



# Function to convert hourly time in slots of 10-minutes
def convert_to_time(frame):
    hours = frame // 6
    minutes = (frame % 6) * 10
    return f"{hours:02d}:{minutes:02d}"


# ------------------------------------------------------------------------------









# ----------------------------------------------------------
#   Sec 1.6.1 -- 4) STACKED ACTIVITY PLOT 
# ----------------------------------------------------------

# Function to create the stacked area plot representing current activities of a selected class during a tested day
def create_plot(
        aggregated_df,
        categories,
        category_english_dict,
        color_map,
        title,
        file_name,
        num_ticks=12,
        num_axis_time_step=144,
        show_title=False,
        fig_size=(7.2, 4.8),
        font_size=11,
        label_font_size=12,
        tick_font_size=10,
        legend_font_size=8,
        title_font_size=12,
        alpha=0.90,
        legend_position="bottom",
        legend_ncol=4,
        save_png=True,
        save_pdf=True):
    """Create a stacked-area profile plot and save it as PNG and/or PDF."""
    if not save_png and not save_pdf:
        raise ValueError(
            "At least one plot output format must be enabled: "
            "save_png=True and/or save_pdf=True."
        )
    if aggregated_df.empty:
        raise ValueError(f"Cannot plot empty dataframe: {file_name}")

    with plt.rc_context({
        "font.size": font_size,
        "axes.labelsize": label_font_size,
        "xtick.labelsize": tick_font_size,
        "ytick.labelsize": tick_font_size,
        "legend.fontsize": legend_font_size,
        "axes.titlesize": title_font_size,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }):
        fig, ax = plt.subplots(figsize=fig_size)
        n_rows = len(aggregated_df)
        x = np.arange(n_rows)
        bottom = np.zeros(n_rows, dtype=float)

        for category_name in categories:
            if category_name not in aggregated_df.columns:
                continue

            values = aggregated_df[category_name].to_numpy(dtype=float)
            ax.fill_between(
                x,
                bottom,
                bottom + values,
                label=category_english_dict.get(category_name, category_name),
                color=color_map.get(category_name, "#999999"),
                alpha=alpha,
                linewidth=0.15,
                edgecolor="white",
            )
            bottom += values

        if isinstance(aggregated_df.index, pd.DatetimeIndex):
            timesteps = [idx.strftime("%H:%M") for idx in aggregated_df.index]
        else:
            try:
                timesteps = [convert_to_time(int(i)) for i in aggregated_df.index]
            except (TypeError, ValueError):
                timesteps = [
                    f"{i // 6:02d}:{(i % 6) * 10:02d}"
                    for i in range(num_axis_time_step)
                ]

        step = max(1, n_rows // num_ticks)
        tick_positions = list(range(0, n_rows, step))
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([timesteps[i] for i in tick_positions])
        ax.set_xlim(0, n_rows - 1)
        ax.set_ylim(0, max(float(bottom.max()) * 1.02, 1.0))
        ax.set_xlabel("Hour of the day")
        ax.set_ylabel("Number of people")

        if show_title:
            ax.set_title(title, pad=10)

        ax.grid(axis="y", linestyle="-", linewidth=0.4, alpha=0.45)
        ax.grid(axis="x", visible=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        handles, labels = ax.get_legend_handles_labels()
        if legend_position == "right":
            ax.legend(
                handles,
                labels,
                loc="center left",
                bbox_to_anchor=(1.02, 0.5),
                frameon=False,
                handlelength=1.2,
                borderaxespad=0.0,
            )
        elif legend_position == "bottom":
            ax.legend(
                handles,
                labels,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.18),
                ncol=legend_ncol,
                frameon=False,
                handlelength=1.2,
                columnspacing=1.0,
                labelspacing=0.6,
            )

        output_path = output_pictures_folder / file_name
        output_path.parent.mkdir(parents=True, exist_ok=True)

        if save_png:
            fig.savefig(
                output_path,
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.04,
            )

        if save_pdf:
            fig.savefig(
                output_path.with_suffix(".pdf"),
                bbox_inches="tight",
                pad_inches=0.04,
            )

        plt.close(fig)

### 1.6.2 Debug and validation helpers

In [ ]:
"""
    This section includes some functions suited for sanity check between new aggregated
    dataframes and the ones provided by the authors (derived from Italian 2013 ISTAT Time-Use-Surveys).

    If new TUS data are available (e.g., Eurostat-based, or more recent), it is useful to check they
    have same structure as the original ones.

    NOTE: Be aware that if you decide to modify the aggregation structure between activities (or locations),
    you have to modify and adjust also the input file "descrizione_dati_TUS.xlsx" (i.e., description of TUS data),
    where you can define the aggregation scheme and the information related to data is provided.

"""


# ----------------------------------------------------------
#       Sec 1.6.2 -- 1)     Optional validation -- Compare rebuilt inputs against legacy inputs
# ----------------------------------------------------------

# ----------------------------------------------------------
# Independent sanity checks and optional regression comparisons
# ----------------------------------------------------------

# Function to compare and evaluate sanity check of input dataframes
def validate_rebuilt_input_dataframes(
        diary_with_individuals_df,
        activity_144_timestep_df,
        place_144_timestep_df):
    """
    Run independent sanity checks on the rebuilt ENG_01 input dataframes.

    These checks do not compare against the legacy outputs.
    They verify whether the rebuilt dataframes are structurally consistent
    with the assumptions used later in ENG_01.
    """
    print("=" * 80)
    print("Independent validation of rebuilt input dataframes")
    print("=" * 80)

    time_step_columns = [
        f"time_step_{i}"
        for i in range(144)
    ]

    # ----------------------
    # 1) Required columns
    # ----------------------
    print("")
    print("1) Checking required columns...")

    required_diary_columns = [
        "persona",
        "Descrizione gsett",
        "Trimestre",
        "rip",
        "tipnu2",
        "rpnuc2"
    ]

    missing_diary_columns = [
        col for col in required_diary_columns
        if col not in diary_with_individuals_df.columns
    ]

    missing_activity_columns = [
        col for col in ["persona"] + time_step_columns
        if col not in activity_144_timestep_df.columns
    ]

    missing_place_columns = [
        col for col in ["persona"] + time_step_columns
        if col not in place_144_timestep_df.columns
    ]

    print(f"Missing diary columns: {missing_diary_columns}")
    print(f"Missing activity timestep columns: {missing_activity_columns[:10]}")
    print(f"Missing place timestep columns: {missing_place_columns[:10]}")

    # ----------------------
    # 2) Quarter and day labels
    # ----------------------
    print("")
    print("2) Checking day-type and quarter labels...")

    expected_day_types = set(source_day_types)
    expected_quarters = set(quarter_abbr_dict.values())

    actual_day_types = set(
        diary_with_individuals_df["Descrizione gsett"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    actual_quarters = set(
        diary_with_individuals_df["Trimestre"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    print(f"Expected day types: {sorted(expected_day_types)}")
    print(f"Actual day types:   {sorted(actual_day_types)}")
    print(f"Unexpected day types: {sorted(actual_day_types - expected_day_types)}")
    print(f"Missing day types:    {sorted(expected_day_types - actual_day_types)}")

    print(f"Expected quarters: {sorted(expected_quarters)}")
    print(f"Actual quarters:   {sorted(actual_quarters)}")
    print(f"Unexpected quarters: {sorted(actual_quarters - expected_quarters)}")
    print(f"Missing quarters:    {sorted(expected_quarters - actual_quarters)}")

    # ----------------------
    # 3) Timestep value ranges
    # ----------------------
    print("")
    print("3) Checking timestep value ranges...")

    activity_values = activity_144_timestep_df[time_step_columns]
    place_values = place_144_timestep_df[time_step_columns]

    activity_min = activity_values.min().min()
    activity_max = activity_values.max().max()
    place_min = place_values.min().min()
    place_max = place_values.max().max()

    print(f"Activity timestep min value: {activity_min}")
    print(f"Activity timestep max value: {activity_max}")
    print(f"Expected activity range: 1 to {len(aggregated_activities)}")

    print(f"Place timestep min value: {place_min}")
    print(f"Place timestep max value: {place_max}")
    print(f"Expected place range: 1 to {len(aggregated_places)}")

    activity_zero_count = int((activity_values == 0).sum().sum())
    place_zero_count = int((place_values == 0).sum().sum())

    print(f"Activity cells equal to 0: {activity_zero_count}")
    print(f"Place cells equal to 0: {place_zero_count}")

    if activity_zero_count > 0:
        print("WARNING: Some activity timestep cells are equal to 0.")
        print("This usually means that at least one activity label was not mapped correctly.")

    if place_zero_count > 0:
        print("WARNING: Some place timestep cells are equal to 0.")
        print("This usually means that at least one place label was not mapped correctly.")

    # ----------------------
    # 4) Person coverage
    # ----------------------
    print("")
    print("4) Checking person coverage across dataframes...")

    diary_people = set(diary_with_individuals_df["persona"].dropna().astype(str).unique())
    activity_people = set(activity_144_timestep_df["persona"].dropna().astype(str).unique())
    place_people = set(place_144_timestep_df["persona"].dropna().astype(str).unique())

    print(f"People in diary + individuals dataframe: {len(diary_people)}")
    print(f"People in activity timestep dataframe:  {len(activity_people)}")
    print(f"People in place timestep dataframe:     {len(place_people)}")

    print(f"People in activity but not in diary: {len(activity_people - diary_people)}")
    print(f"People in place but not in diary:    {len(place_people - diary_people)}")
    print(f"People in diary but not in activity: {len(diary_people - activity_people)}")
    print(f"People in diary but not in place:    {len(diary_people - place_people)}")

    # ----------------------
    # 5) NaN checks on mapped columns
    # ----------------------
    print("")
    print("5) Checking missing values in key mapped columns...")

    key_mapped_columns = [
        "Attività aggregata",
        "Luogo aggregato",
        "Trimestre",
        "Descrizione gsett"
    ]

    for column in key_mapped_columns:
        if column in diary_with_individuals_df.columns:
            missing_count = int(diary_with_individuals_df[column].isna().sum())
            print(f"{column}: {missing_count} missing values")
        else:
            print(f"{column}: column not found")

    print("")
    print("Independent validation completed.")
    print("=" * 80)



# Function to compare df_diario_giornaliero_full between old and new configuration
def compare_dataframes_ignoring_row_order(old_df, new_df, dataframe_name):
    """
    Compare two dataframes as unordered collections of rows.

    This is useful when the same rows may appear in a different order.
    """
    print("=" * 80)
    print(f"Comparing dataframe ignoring row order: {dataframe_name}")
    print("=" * 80)

    if old_df.shape != new_df.shape:
        print(f"Different shape: old={old_df.shape}, new={new_df.shape}")
        return False

    if list(old_df.columns) != list(new_df.columns):
        print("Different columns or different column order.")
        return False

    old_normalized = old_df.copy()
    new_normalized = new_df.copy()

    # Convert all columns to string-compatible values for robust row hashing.
    # NaN values are replaced by a stable placeholder.
    old_normalized = old_normalized.fillna("__NA__").astype(str)
    new_normalized = new_normalized.fillna("__NA__").astype(str)

    old_hashes = pd.util.hash_pandas_object(
        old_normalized,
        index=False
    ).value_counts().sort_index()

    new_hashes = pd.util.hash_pandas_object(
        new_normalized,
        index=False
    ).value_counts().sort_index()

    try:
        pd.testing.assert_series_equal(
            old_hashes,
            new_hashes,
            check_names=False
        )

        print("RESULT: SAME ROWS, POSSIBLY DIFFERENT ORDER")
        return True

    except AssertionError as error:
        print("RESULT: DIFFERENT ROW SET")
        print("")
        print(error)
        return False


# Function to compare people sets within "df_diario_giornaliero_full"
def compare_people_sets_by_filter_group(legacy_df, rebuilt_df):
    group_cols = [
        "Descrizione gsett",
        "Trimestre",
        "rip",
        "tipnu2",
        "rpnuc2",
    ]

    legacy_groups = (
        legacy_df
        .groupby(group_cols)["persona"]
        .apply(lambda x: frozenset(x.dropna().astype(str).unique()))
        .reset_index(name="legacy_people")
    )

    rebuilt_groups = (
        rebuilt_df
        .groupby(group_cols)["persona"]
        .apply(lambda x: frozenset(x.dropna().astype(str).unique()))
        .reset_index(name="rebuilt_people")
    )

    comparison = pd.merge(
        legacy_groups,
        rebuilt_groups,
        on=group_cols,
        how="outer"
    )

    comparison["same_people"] = (
        comparison["legacy_people"] == comparison["rebuilt_people"]
    )

    different_groups = comparison[~comparison["same_people"]]

    print(f"Number of groups with different people sets: {len(different_groups)}")

    if len(different_groups) > 0:
        print(different_groups[group_cols].head(30))

    return different_groups



# Function to run the optional debugging phase for comparing rebuilt aggregated dataframes with the existing ones
def run_input_debugging(
        diary_with_individuals_df,
        activity_144_timestep_df,
        place_144_timestep_full_df):
    """Run regression and structural checks on rebuilt inputs."""
    print("=" * 80)
    print("OPTIONAL DEBUGGING OF REBUILT INPUT DATAFRAMES")
    print("=" * 80)

    legacy_path = legacy_input_folder / "df_diario_giorn_con_individui_full.pkl.gz"
    rebuilt_path = input_folder / "df_diario_giorn_con_individui_full.pkl.gz"

    if not legacy_path.exists():
        print(f"Legacy comparison file not found: {legacy_path}")
        print("Running independent validation only.")
    else:
        legacy_diary_df = load_pickle_gzip(legacy_path)
        rebuilt_diary_df = load_pickle_gzip(rebuilt_path)
        print(f"Legacy diary shape:  {legacy_diary_df.shape}")
        print(f"Rebuilt diary shape: {rebuilt_diary_df.shape}")

        if RUN_STRICT_ROWSET_DEBUG:
            compare_dataframes_ignoring_row_order(
                old_df=legacy_diary_df,
                new_df=rebuilt_diary_df,
                dataframe_name="df_diario_giorn_con_individui_full.pkl.gz",
            )

        different_people_groups = compare_people_sets_by_filter_group(
            legacy_diary_df,
            rebuilt_diary_df,
        )
        if different_people_groups.empty:
            print("OK: every operational filter group contains the same people.")
        else:
            print("WARNING: some operational filter groups contain different people.")
            print(different_people_groups.head(30))

    validate_rebuilt_input_dataframes(
        diary_with_individuals_df=diary_with_individuals_df,
        activity_144_timestep_df=activity_144_timestep_df,
        place_144_timestep_df=place_144_timestep_full_df,
    )

### 1.6.3 F(x) Annual and weekly profiles

In [ ]:
# ----------------------------------------------------------
#   Sec 1.6.3 -- 1) Annual and weekly profiles
# ----------------------------------------------------------

# Function to read a code description robustly from Excel columns with mixed data types.
def lookup_description(description_df, code_column, description_column, code_value):
    """Read a description robustly when Excel code columns use mixed dtypes."""
    if code_column not in description_df.columns:
        return str(code_value)
    if description_column not in description_df.columns:
        return str(code_value)

    normalized_codes = normalize_mapping_code(
        description_df[code_column],
        strip_leading_zeros=True,
    )
    target_code = normalize_mapping_code(
        pd.Series([code_value]),
        strip_leading_zeros=True,
    ).iloc[0]

    matches = description_df.loc[normalized_codes == target_code, description_column]
    if matches.empty:
        return str(code_value)
    return str(matches.iloc[0])


# Function to build and validate the full-year datetime index at 10-minute resolution.
def build_simulation_datetime_index():
    """Build and validate the full-year 10-minute datetime index."""
    expected_days = 366 if pd.Timestamp(f"{simulation_year}-01-01").is_leap_year else 365
    if simulation_days != expected_days:
        raise ValueError(
            f"simulation_days={simulation_days} is inconsistent with "
            f"simulation_year={simulation_year} ({expected_days} days)."
        )

    return pd.date_range(
        start=f"{simulation_year}-01-01 00:00:00",
        periods=num_time_steps * simulation_days,
        freq="10min",
    )


# Function to generate annual synthetic profiles using the original person-day-timestep loop order.
def simulate_annual_profiles_original_order(
        datetime_index,
        tipnu2_value,
        rpnuc2_value,
        matrix_area,
        input_matrices_folder):
    """
    Generate annual profiles with the same person -> day -> timestep loop as v01.

    The final state of one day is reused as the first state of the following day
    when CONTINUE_STATE_ACROSS_DAYS is True, matching the original procedure.
    """
    total_steps = simulation_days * num_time_steps
    states = np.arange(1, len(aggregated_activities) + 1)
    profiles = np.empty((simulated_people, total_steps), dtype=np.int16)

    for person_index in tqdm(
            range(simulated_people),
            desc=f"Simulating {matrix_area}"):
        previous_state = None
        write_index = 0

        for day_index in range(simulation_days):
            date = datetime_index[day_index * num_time_steps]
            matrix, initial_probabilities = load_transition_cached(
                date.day_name(),
                get_quarter(date.month),
                tipnu2_value,
                rpnuc2_value,
                matrix_area,
                input_matrices_folder,
            )

            if previous_state is None or not CONTINUE_STATE_ACROSS_DAYS:
                current_state = np.random.choice(
                    states,
                    p=initial_probabilities[:, 0],
                )
            else:
                current_state = previous_state

            profiles[person_index, write_index] = current_state

            for timestep in range(1, num_time_steps):
                transition_probabilities = matrix[
                    timestep - 1,
                    current_state - 1,
                ]
                current_state = np.random.choice(
                    states,
                    p=transition_probabilities,
                )
                profiles[person_index, write_index + timestep] = current_state

            previous_state = current_state
            write_index += num_time_steps

    return pd.DataFrame(
        profiles.T,
        index=datetime_index,
        columns=[f"person_{index + 1}" for index in range(simulated_people)],
    )


# Function to build the compact English filename used for annual synthetic activity profiles.
def build_yearly_profile_file_name(
        region_abbr,
        tipnu2_value,
        rpnuc2_value):

    profile_type = suffix_english_dict[
        (tipnu2_value, rpnuc2_value)
    ]

    return (
        f"synthetic_activity_profiles_"
        f"{simulation_year}_{simulated_people}pp_"
        f"{region_abbr}_"
        f"householdtype_{tipnu2_value}_"
        f"member_{rpnuc2_value}_"
        f"{profile_type}.xlsx"
    )


# Return the identifiers of the currently active execution stages.
def active_stage_groups():
    stage_flags = {
        "optional_rebuild": RUN_AGGR_DATASETS,
        "optional_debug": RUN_DEBUG_INPUTS,
        "matrices_plots": RUN_MATRICES or RUN_PLOTS,
        "profile_pipeline": RUN_YEARLY_PROFILES or RUN_WEEKLY_PROFILES,
    }

    return [
        stage_id
        for stage_id, enabled in stage_flags.items()
        if enabled
    ]


# Check that only the intended execution stage is enabled.
def validate_stage_selection(expected_stage=None):
    active = active_stage_groups()

    if len(active) > 1:
        active_labels = [
            STAGE_LABELS[stage_id]
            for stage_id in active
        ]

        raise RuntimeError(
            "More than one MODULE_01 stage is active:\n - "
            + "\n - ".join(active_labels)
            + "\nEnable only one stage at a time."
        )

    if expected_stage is not None and active and active[0] != expected_stage:
        raise RuntimeError(
            f"This cell belongs to '{STAGE_LABELS[expected_stage]}', "
            f"but the active stage is '{STAGE_LABELS[active[0]]}'. "
            "Review the flags in Section 1.2."
        )

    return active


# Function for collect information from activity profile dataset and Extract profile metadata from new English, previous English and legacy Italian filenames.
def parse_activity_profile_filename(file_name):
    name_body = Path(file_name).stem

    filename_patterns = [
        # New compact English convention.
        re.compile(
            r"^synthetic_activity_profiles_"
            r"(?P<year>\d{4})_"
            r"(?P<people>\d+)pp_"
            r"(?P<region>.+?)_"
            r"householdtype_(?P<household_type>\d+)_"
            r"member_(?P<member>\d+)_"
            r"(?P<suffix>.+)$"
        ),

        # Previous v05 English convention.
        re.compile(
            r"^synthetic_activity_profiles_"
            r"(?P<year>\d{4})_"
            r"(?P<people>\d+)_pp_"
            r"region_(?P<region>.+?)_"
            r"householdtype_(?P<household_type>\d+)_"
            r"member_(?P<member>\d+)_"
            r"(?P<suffix>.+)$"
        ),

        # Original Italian convention.
        re.compile(
            r"^profili_sintetici_attività_"
            r"(?P<year>\d{4})_"
            r"(?P<people>\d+)_pp_"
            r"reg_(?P<region>.+?)_"
            r"tipnuc_(?P<household_type>\d+)_"
            r"membro_(?P<member>\d+)_"
            r"(?P<suffix>.+)$"
        ),
    ]

    for pattern in filename_patterns:
        match = pattern.match(name_body)

        if match:
            values = match.groupdict()

            return {
                "name_body": name_body,
                "year": int(values["year"]),
                "simulated_people": int(values["people"]),
                "region": values["region"],
                "tipnu2_value": int(values["household_type"]),
                "rpnuc2_value": int(values["member"]),
                "suffix": values["suffix"],
            }

    raise ValueError(
        "Could not parse annual activity-profile filename:\n"
        f"  {file_name}"
    )



# Function for builduing a correct name between Italian or English dataframes input files.
def build_weekly_profile_file_name(profile_info):
    tipnu2_value = profile_info["tipnu2_value"]
    rpnuc2_value = profile_info["rpnuc2_value"]

    profile_type = suffix_english_dict.get(
        (tipnu2_value, rpnuc2_value),
        profile_info["suffix"],
    )

    return (
        f"weekly_activity_profiles_"
        f"{profile_info['year']}_"
        f"{profile_info['simulated_people']}pp_"
        f"{profile_info['region']}_"
        f"householdtype_{tipnu2_value}_"
        f"member_{rpnuc2_value}_"
        f"{profile_type}.xlsx"
    )


# Function to read an annual profile Excel file using either English or legacy Italian sheet names.
def read_annual_profile_excel(input_path):
    """
    Read an annual synthetic profile Excel file.

    Supports both:
        - English sheet name: Profiles
        - Italian sheet name: Profili

    If neither is found, the first sheet is used.
    """
    excel_file = pd.ExcelFile(input_path)

    if "Profiles" in excel_file.sheet_names:
        profile_sheet = "Profiles"
    elif "Profili" in excel_file.sheet_names:
        profile_sheet = "Profili"
    else:
        profile_sheet = excel_file.sheet_names[0]

    df = pd.read_excel(
        input_path,
        sheet_name=profile_sheet,
        parse_dates=["Datetime"]
    )

    if "Datetime" not in df.columns:
        raise ValueError(
            f"'Datetime' column not found in {input_path}, sheet '{profile_sheet}'"
        )

    df = df.set_index("Datetime")

    return df

## 1.7 Execution-plan validation

This lightweight cell validates the hybrid policy and checks that only one independent heavy stage is enabled. This avoids the risk to execute multiple computationally demanding stages during the same simulation. The same validation is repeated at the beginning of every executable section. It returns which stage is going to be processed in the current run.


In [ ]:
validate_matrix_geographical_policies()
active = validate_stage_selection()

print("Selected MODULE_01 execution stage:")

if active:
    print(f" - {STAGE_LABELS[active[0]]}")
else:
    print(" - none")


## 1.8 (Optional) rebuild of the aggregated ISTAT input dataframes

This section performs only the source-data reconstruction. Starting from the original (Italian ISTAT) TUS database, it returns the dataframes with aggregated and reorganized information (e.g., grouping similar activities ("reading newspaper" + "reading books" --> "reading")). The resulting dataframes are already provided in the input folder (i.e., "01_input_profiles_att"), so this stage is just optional: run it alone only if alternative grouping assumptions or any modification of the input files is required. 

Computationally demanding, so run it alone, then restart the kernel before calculating matrices or generating profiles. Missing input files do not trigger this stage automatically.


In [ ]:
if RUN_AGGR_DATASETS:
    validate_stage_selection("optional_rebuild")
    stage_start_time = datetime.datetime.now()
    print(f"Input-rebuild stage started: {stage_start_time}")

    rebuild_input_dataframes(input_folder)

    gc.collect()
    stage_end_time = datetime.datetime.now()
    print(f"Input-rebuild stage ended: {stage_end_time}")
    print(f"Stage execution time: {stage_end_time - stage_start_time}")
else:
    print("Section 1.8 skipped: aggregated inputs are not rebuilt.")


## 1.9 (Optional) validation of rebuilt input dataframes

This section loads the optional rebuilt aggregated dataframes in section 1.8 and compares them with the ones provided by the authors. Computationally demanding, it has therefore be executed independently from reconstruction and downstream calculations.


In [ ]:
if RUN_DEBUG_INPUTS:
    validate_stage_selection("optional_debug")
    stage_start_time = datetime.datetime.now()
    print(f"Input-debug stage started: {stage_start_time}")

    debug_inputs = load_required_inputs(
        need_diary=True,
        need_activity=True,
        need_place=True,
    )
    run_input_debugging(
        diary_with_individuals_df=debug_inputs["diary"],
        activity_144_timestep_df=debug_inputs["activity"],
        place_144_timestep_full_df=debug_inputs["place"],
    )

    del debug_inputs
    gc.collect()
    stage_end_time = datetime.datetime.now()
    print(f"Input-debug stage ended: {stage_end_time}")
    print(f"Stage execution time: {stage_end_time - stage_start_time}")
else:
    print("Section 1.9 skipped: input debugging is not requested.")


## 1.10 Build transition matrices, initial probabilities, and diagnostic plots

For each individual configuration, it calculates the corresponding transition matrix, initial probability, and plot calculations, taking into account the considered days, quarters/season, and geographical area.

In [ ]:
if RUN_MATRICES or RUN_PLOTS:
    validate_stage_selection("matrices_plots")
    stage_start_time = datetime.datetime.now()
    print(f"Matrix/plot stage started: {stage_start_time}")

    need_activity_data = (
        (RUN_MATRICES and BUILD_ACTIVITY_MATRICES)
        or (RUN_PLOTS and PLOT_ACTIVITIES)
    )
    need_place_data = (
        (RUN_MATRICES and BUILD_PLACE_MATRICES)
        or (RUN_PLOTS and PLOT_PLACES)
    )

    matrix_inputs = load_required_inputs(
        need_diary=True,
        need_activity=need_activity_data,
        need_place=need_place_data,
    )
    diary_with_individuals_df = matrix_inputs["diary"]
    activity_144_timestep_df = matrix_inputs.get("activity")
    place_144_timestep_full_df = matrix_inputs.get("place")

    case_records = []

    for tipnu2_value, rpnuc2_value in individual_configurations:
        matrix_policy = get_matrix_policy_name(tipnu2_value, rpnuc2_value)
        household_types = [tipnu2_value]
        household_positions = [rpnuc2_value]

        print("=" * 80)
        print(
            f"Configuration ({tipnu2_value}, {rpnuc2_value}) uses policy "
            f"'{matrix_policy}'."
        )

        # The policy table explicitly decides whether this configuration uses
        # five distinct regions, two macro-areas, or only South+Islands pooled.
        for area_case in get_matrix_area_cases(tipnu2_value, rpnuc2_value):
            area_abbr = area_case["area_abbr"]
            region_codes = area_case["region_codes"]

            activity_configuration_folder = None
            place_configuration_folder = None
            if RUN_MATRICES and BUILD_ACTIVITY_MATRICES:
                activity_configuration_folder = configuration_folder(
                    activity_matrices_folder,
                    area_abbr,
                    tipnu2_value,
                    rpnuc2_value,
                )
            if RUN_MATRICES and BUILD_PLACE_MATRICES:
                place_configuration_folder = configuration_folder(
                    location_matrices_folder,
                    area_abbr,
                    tipnu2_value,
                    rpnuc2_value,
                )

            print("-" * 80)
            print(
                f"Matrix area: {area_abbr}; source ISTAT regions: {region_codes}"
            )

            for quarter, day_type in product(source_quarters, source_day_types):
                quarter_abbr = quarter_abbr_dict[quarter]

                plot_case_selected = (
                    RUN_PLOTS
                    and (
                        PLOT_SELECTED_CONFIGURATIONS is None
                        or (tipnu2_value, rpnuc2_value)
                        in PLOT_SELECTED_CONFIGURATIONS
                    )
                    and (
                        PLOT_SELECTED_AREAS is None
                        or area_abbr in PLOT_SELECTED_AREAS
                    )
                    and (
                        PLOT_SELECTED_QUARTERS is None
                        or quarter_abbr in PLOT_SELECTED_QUARTERS
                    )
                    and (
                        PLOT_SELECTED_DAY_TYPES is None
                        or day_type in PLOT_SELECTED_DAY_TYPES
                    )
                )

                # When only plots are requested, skip unselected cases before
                # filtering the large source DataFrames.
                if not RUN_MATRICES and not plot_case_selected:
                    continue


                filtered_diary_df = apply_filters(
                    diary_with_individuals_df,
                    day_type,
                    quarter_abbr,
                    region_codes,
                    household_types,
                    household_positions,
                )

                filtered_people = filtered_diary_df["persona"].unique()
                num_individuals = len(filtered_people)
                has_data = num_individuals > 0
                below_warning_threshold = (
                    has_data and num_individuals < MIN_SOURCE_RESPONDENTS_WARNING
                )

                case_records.append({
                    "matrix_policy": matrix_policy,
                    "matrix_area": area_abbr,
                    "source_region_codes": ",".join(region_codes),
                    "household_type": tipnu2_value,
                    "household_position": rpnuc2_value,
                    "quarter": quarter_abbr,
                    "day_type": day_type,
                    "num_individuals": num_individuals,
                    "has_data": has_data,
                    "below_warning_threshold": below_warning_threshold,
                })

                print(
                    f"{area_abbr} | ({tipnu2_value},{rpnuc2_value}) | "
                    f"{quarter_abbr} | {day_type}: {num_individuals} respondents"
                )

                if not has_data:
                    print("No source respondents: case skipped.")
                    continue
                if below_warning_threshold:
                    print(
                        "WARNING: the sample is below "
                        f"MIN_SOURCE_RESPONDENTS_WARNING="
                        f"{MIN_SOURCE_RESPONDENTS_WARNING}."
                    )

                filtered_activity_df = None
                filtered_place_df = None
                aggregated_activity_df = None
                aggregated_place_df = None

                if need_activity_data:
                    filtered_activity_df = activity_144_timestep_df[
                        activity_144_timestep_df["persona"].isin(filtered_people)
                    ]
                    if plot_case_selected and PLOT_ACTIVITIES:
                        aggregated_activity_df = create_aggregated_df_vectorized(
                            filtered_activity_df,
                            aggregated_activities,
                        )

                if need_place_data:
                    filtered_place_df = place_144_timestep_full_df[
                        place_144_timestep_full_df["persona"].isin(filtered_people)
                    ]
                    if plot_case_selected and PLOT_PLACES:
                        aggregated_place_df = create_aggregated_df_vectorized(
                            filtered_place_df,
                            aggregated_places,
                        )

                if plot_case_selected:
                    area_suffix = (
                        f"_{area_abbr}" if INCLUDE_AREA_IN_PLOT_FILENAME else ""
                    )
                    day_label = day_english_dict[day_type]
                    quarter_label = quarter_abbr_english_dict[quarter_abbr]
                    family_label = suffix_english_dict[(tipnu2_value, rpnuc2_value)]
                    plot_specs = []

                    if PLOT_ACTIVITIES:
                        plot_specs.append({
                            "data": aggregated_activity_df,
                            "categories": aggregated_activities,
                            "labels": activity_english_dict,
                            "colors": activity_color_map,
                            "title": (
                                f"Aggregated daily activities - {day_label} - "
                                f"{quarter_label} | Area: {area_abbr} | "
                                f"Family type: {family_label}"
                            ),
                            "base_name": (
                                f"activities_{day_label}_{quarter_label}_"
                                f"householdtype_{tipnu2_value}_member_"
                                f"{rpnuc2_value}_ENG{area_suffix}"
                            ),
                        })

                    if PLOT_PLACES:
                        plot_specs.append({
                            "data": aggregated_place_df,
                            "categories": aggregated_places,
                            "labels": place_english_dict,
                            "colors": place_color_map,
                            "title": (
                                f"Aggregated occupied places - {day_label} - "
                                f"{quarter_label} | Area: {area_abbr} | "
                                f"Family type: {family_label}"
                            ),
                            "base_name": (
                                f"places_{day_label}_{quarter_label}_"
                                f"householdtype_{tipnu2_value}_member_"
                                f"{rpnuc2_value}_ENG{area_suffix}"
                            ),
                        })

                    for plot_spec in plot_specs:
                        for layout in PLOT_LAYOUTS:
                            if layout not in PLOT_STYLE_PRESETS:
                                raise ValueError(
                                    f"Unknown plot layout '{layout}'. "
                                    f"Choose from {tuple(PLOT_STYLE_PRESETS)}."
                                )
                            create_plot(
                                aggregated_df=plot_spec["data"],
                                categories=plot_spec["categories"],
                                category_english_dict=plot_spec["labels"],
                                color_map=plot_spec["colors"],
                                title=plot_spec["title"],
                                file_name=(
                                    f"{plot_spec['base_name']}_{layout}.png"
                                ),
                                save_png=PLOT_SAVE_PNG,
                                save_pdf=PLOT_SAVE_PDF,
                                **PLOT_STYLE_PRESETS[layout],
                            )

                if RUN_MATRICES and BUILD_ACTIVITY_MATRICES:
                    activity_matrix, activity_initial_prob = (
                        calculate_transition_matrices_and_initial_probabilities(
                            filtered_activity_df,
                            aggregated_activities,
                            area_abbr,
                            day_type,
                            quarter_abbr,
                            num_time_steps,
                        )
                    )
                    np.save(
                        activity_configuration_folder
                        / build_matrix_file_name(
                            "transition_matrix_activity",
                            area_abbr,
                            day_type,
                            quarter_abbr,
                            tipnu2_value,
                            rpnuc2_value,
                        ),
                        activity_matrix,
                    )
                    np.save(
                        activity_configuration_folder
                        / build_matrix_file_name(
                            "initial_prob_activity",
                            area_abbr,
                            day_type,
                            quarter_abbr,
                            tipnu2_value,
                            rpnuc2_value,
                        ),
                        activity_initial_prob,
                    )

                if RUN_MATRICES and BUILD_PLACE_MATRICES:
                    place_matrix, place_initial_prob = (
                        calculate_transition_matrices_and_initial_probabilities(
                            filtered_place_df,
                            aggregated_places,
                            area_abbr,
                            day_type,
                            quarter_abbr,
                            num_time_steps,
                        )
                    )
                    np.save(
                        place_configuration_folder
                        / build_matrix_file_name(
                            "transition_matrix_places",
                            area_abbr,
                            day_type,
                            quarter_abbr,
                            tipnu2_value,
                            rpnuc2_value,
                        ),
                        place_matrix,
                    )
                    np.save(
                        place_configuration_folder
                        / build_matrix_file_name(
                            "initial_prob_places",
                            area_abbr,
                            day_type,
                            quarter_abbr,
                            tipnu2_value,
                            rpnuc2_value,
                        ),
                        place_initial_prob,
                    )

    case_summary_df = pd.DataFrame(case_records)
    if SAVE_CASE_SUMMARY and not case_summary_df.empty:
        case_summary_path = diagnostics_folder / "module_01_source_case_summary.csv"
        case_summary_df.to_csv(case_summary_path, index=False)
        print(f"Saved source-case summary: {case_summary_path}")

    del matrix_inputs
    gc.collect()
    stage_end_time = datetime.datetime.now()
    print(f"Matrix/plot stage ended: {stage_end_time}")
    print(f"Stage execution time: {stage_end_time - stage_start_time}")
else:
    print("Section 1.10 skipped: matrices and plots are not requested.")


## 1.11 Create annual synthetic activity profiles

This section creates annual synthetic activity profiles for the considered individual classes, by using the previously reconstructed information (i.e., initial probabilities, transition matrices, description input tables)

In [ ]:
if RUN_YEARLY_PROFILES:
    validate_stage_selection("profile_pipeline")
    stage_start_time = datetime.datetime.now()
    print(f"Annual-profile stage started: {stage_start_time}")

    if RANDOM_SEED is not None:
        np.random.seed(RANDOM_SEED)

    description_dict = load_description_dictionary()
    datetime_index = build_simulation_datetime_index()
    print(f"January 1 {simulation_year} is a {datetime_index[0].day_name()}.")

    for tipnu2_value, rpnuc2_value in individual_configurations:
        household_description = lookup_description(
            description_dict,
            "tipnu2",
            "tipo di nucleo",
            tipnu2_value,
        )
        member_description = lookup_description(
            description_dict,
            "rpnuc2",
            "posizione nel nucleo familiare",
            rpnuc2_value,
        )
        matrix_policy = get_matrix_policy_name(tipnu2_value, rpnuc2_value)

        print("=" * 80)
        print(
            f"Generating annual profiles for ({tipnu2_value}, {rpnuc2_value}); "
            f"matrix policy='{matrix_policy}'."
        )

        for region_code, region_name in italy_regions_dict.items():
            region_abbr = region_abbr_dict[region_name]

            # The same policy table used in Section 1.8 maps each requested
            # regional profile to the matrix area created for this class.
            matrix_area = resolve_matrix_area(
                region_code,
                tipnu2_value,
                rpnuc2_value,
            )
            input_matrices_folder = (
                activity_matrices_folder
                / matrix_area
                / f"householdtype_{tipnu2_value}_member_{rpnuc2_value}"
            )

            print(
                f"Output region {region_abbr} uses matrices from "
                f"'{matrix_area}'."
            )

            profile_df = simulate_annual_profiles_original_order(
                datetime_index=datetime_index,
                tipnu2_value=tipnu2_value,
                rpnuc2_value=rpnuc2_value,
                matrix_area=matrix_area,
                input_matrices_folder=input_matrices_folder,
            )

            profile_saving_folder = yearly_activity_profiles_folder / region_abbr
            profile_saving_folder.mkdir(parents=True, exist_ok=True)
            excel_path = profile_saving_folder / build_yearly_profile_file_name(
                region_abbr,
                tipnu2_value,
                rpnuc2_value,
            )

            info_df = pd.DataFrame({
                "Description": [
                    "Simulated year",
                    "Number of people for whom a synthetic profile was generated",
                    "Family category",
                    "Family member",
                    "Region considered",
                    "Matrix geographical policy",
                    "Matrix area used",
                    "Continue state across days",
                    "Random seed",
                ],
                "Value": [
                    simulation_year,
                    simulated_people,
                    household_description,
                    member_description,
                    region_abbr,
                    matrix_policy,
                    matrix_area,
                    CONTINUE_STATE_ACROSS_DAYS,
                    RANDOM_SEED if RANDOM_SEED is not None else "random",
                ],
                "Code": [
                    "",
                    "",
                    tipnu2_value,
                    rpnuc2_value,
                    "",
                    "",
                    "",
                    "",
                    "",
                ],
            })

            with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
                profile_df.to_excel(
                    writer,
                    sheet_name="Profiles",
                    index_label="Datetime",
                )
                info_df.to_excel(writer, sheet_name="Information", index=False)
                writer.sheets["Information"].cell(
                    row=len(info_df) + 3,
                    column=1,
                ).value = (
                    f"The profiles belong to family category "
                    f"'{household_description}', member '{member_description}', "
                    f"region {region_abbr}, using matrices from {matrix_area}."
                )

            print(f"Saved annual profile: {excel_path}")

            # Release the completed regional profile before the next region.
            del profile_df
            del info_df
            gc.collect()


        # Clear cached matrices before moving to the following individual class.
        load_transition_cached.cache_clear()
        gc.collect()

    del description_dict
    gc.collect()
    stage_end_time = datetime.datetime.now()
    print(f"Annual-profile stage ended: {stage_end_time}")
    print(f"Stage execution time: {stage_end_time - stage_start_time}")
else:
    print("Section 1.10 skipped: annual profiles are not generated.")


## 1.12 Create weekly activity profiles for selected periods

This section reads existing annual activity profiles (previously generated) and extracts the selected representative periods. These are going to be used by the following modules.


In [ ]:
if RUN_WEEKLY_PROFILES:
    validate_stage_selection("profile_pipeline")
    stage_start_time = datetime.datetime.now()
    print(f"Weekly-extraction stage started: {stage_start_time}")

    for region_name, region_abbr in region_abbr_dict.items():
        input_dir = yearly_activity_profiles_folder / region_abbr
        if not input_dir.is_dir():
            print(f"Folder not found for {region_name}: {input_dir}; skipped.")
            continue


        candidate_files = sorted({
            *input_dir.glob("synthetic_activity_profiles_*.xlsx"),
            *input_dir.glob("profili_sintetici_attività_*.xlsx"),
        })

        # Keep one annual file per profile, preferring the newest English convention.
        preferred_files = {}

        for candidate_path in candidate_files:
            try:
                candidate_info = parse_activity_profile_filename(
                    candidate_path.name
                )
            except ValueError as error:
                print(f"Unrecognized annual profile skipped: {error}")
                continue

            configuration = (
                candidate_info["tipnu2_value"],
                candidate_info["rpnuc2_value"],
            )

            if configuration not in individual_configurations:
                continue

            if candidate_info["year"] != simulation_year:
                continue

            if candidate_info["simulated_people"] != simulated_people:
                continue

            profile_key = (
                candidate_info["region"],
                candidate_info["tipnu2_value"],
                candidate_info["rpnuc2_value"],
                candidate_info["year"],
                candidate_info["simulated_people"],
            )

            new_english_prefix = (
                f"synthetic_activity_profiles_"
                f"{candidate_info['year']}_"
                f"{candidate_info['simulated_people']}pp_"
            )

            if candidate_path.name.startswith(new_english_prefix):
                naming_priority = 2       # new compact English name
            elif candidate_path.name.startswith("synthetic_activity_profiles_"):
                naming_priority = 1       # previous English name
            else:
                naming_priority = 0       # legacy Italian name

            current_file = preferred_files.get(profile_key)

            if current_file is None or naming_priority > current_file[0]:
                preferred_files[profile_key] = (
                    naming_priority,
                    candidate_path,
                )

        annual_files = sorted(
            file_path
            for _, file_path in preferred_files.values()
        )




        if not annual_files:
            print(f"No annual profile file found in {input_dir}")
            continue

        output_dir_region = weekly_activity_profiles_folder / region_abbr
        output_dir_region.mkdir(parents=True, exist_ok=True)

        for input_path in tqdm(annual_files, desc=f"Region {region_abbr}"):
            try:
                profile_info = parse_activity_profile_filename(input_path.name)
                tipnu2_value = profile_info["tipnu2_value"]
                rpnuc2_value = profile_info["rpnuc2_value"]
                annual_df = read_annual_profile_excel(input_path)

                output_path = output_dir_region / build_weekly_profile_file_name(
                    profile_info
                )
                if len(str(output_path)) > 240:
                    print(f"Path too long ({len(str(output_path))}); skipped.")
                    continue

                with pd.ExcelWriter(
                        output_path,
                        engine="openpyxl",
                        datetime_format="yyyy-mm-dd HH:MM:SS") as writer:
                    for sheet_name, (start_date, end_date) in selected_periods.items():
                        period_df = annual_df.loc[start_date:end_date]
                        if period_df.empty:
                            print(
                                f"No data for '{sheet_name}' in {input_path.name}"
                            )
                        else:
                            period_df.to_excel(writer, sheet_name=sheet_name[:31])

                    type_description = suffix_english_dict.get(
                        (tipnu2_value, rpnuc2_value),
                        profile_info["suffix"],
                    )
                    info_df = pd.DataFrame({
                        "Description": [
                            "Simulated year",
                            "Simulated people",
                            "Family category",
                            "Family member",
                            "Region considered",
                            "Source annual profile file",
                        ],
                        "Value": [
                            simulation_year,
                            simulated_people,
                            type_description,
                            type_description.split("_")[-1],
                            region_abbr,
                            input_path.name,
                        ],
                        "Code": [
                            "",
                            "",
                            tipnu2_value,
                            rpnuc2_value,
                            "",
                            "",
                        ],
                    })
                    info_df.to_excel(writer, sheet_name="Information", index=False)

                print(f"Saved weekly profile: {output_path}")

            except Exception as error:
                print(f"Error processing {input_path}: {error}")

    stage_end_time = datetime.datetime.now()
    print(f"Weekly-extraction stage ended: {stage_end_time}")
    print(f"Stage execution time: {stage_end_time - stage_start_time}")
else:
    print("Section 1.12 skipped: weekly profiles are not extracted.")


## Conclusion

In [ ]:
#---------------------------------------------------------------------------------------------------------------------------------------------

# Get the end time
end_time = datetime.datetime.now()
print(f"End time : {end_time}")

# Calculate execution time
execution_time = end_time - start_time
print(f"Execution time : {execution_time}")
